# Semantic Model Optimization Scanner — V2.6.1

This notebook finds semantic-model optimization opportunities and writes the results to the attached Lakehouse. It does **not** change the scanned model and does not claim CU savings; CU improvement is measured later after an approved change.

## Run it

1. Attach the output Lakehouse to this notebook.
2. Enter the IDs in **Member inputs**.
3. Select **Run all**, or call the notebook from a Fabric Pipeline.

| Input | What to enter |
|---|---|
| `workspace_ids` | Required. One or more Workspace IDs separated by commas, semicolons, spaces, or new lines. |
| `model_ids_optional` | Optional. Leave blank to scan every semantic model in the listed workspaces; otherwise enter one or more Model IDs. |
| `requested_by_upn_optional` | Optional requester email for the audit log. |
| Runtime dependencies | Supplied by the attached `SMO_Scanner_Environment`; no per-run package installation is required. |

Examples:

```text
workspace_ids = "workspace-id-1, workspace-id-2"
model_ids_optional = ""                         # all models in both workspaces
model_ids_optional = "model-id-1, model-id-2"  # selected models only
```

The platform owner configures authentication and safeguards once in the collapsed **Platform setup** cell. The default `workspace_user` profile uses only the current identity and workspace-scoped APIs. The optional `governance_admin` profile adds an item-access snapshot and therefore requires Fabric Admin API permission. For an SPN run, the target workspace owner must grant the scanner identity access before the scan.


In [ ]:
# Discover modules only; compatibility is based on callable APIs, not exact versions.
from importlib.util import find_spec

REQUIRED_RUNTIME_MODULES = ("sempy.fabric", "sempy_labs")


def module_available(module_name):
    try:
        return find_spec(module_name) is not None
    except (ImportError, ModuleNotFoundError):
        return False


missing_runtime_modules = [
    module_name for module_name in REQUIRED_RUNTIME_MODULES if not module_available(module_name)
]
if missing_runtime_modules:
    raise RuntimeError(
        "Scanner runtime is missing required modules: "
        + ", ".join(missing_runtime_modules)
        + ". Publish the attached SMO_Scanner_Environment and start a fresh session."
    )
print("Scanner runtime modules discovered; callable capability validation follows.")


In [ ]:
# MEMBER INPUTS — these are the only values auto-populated in a Pipeline Notebook activity.

workspace_ids = ""                 # Required: id1,id2 or one ID per line
model_ids_optional = ""             # Optional: blank = every model in the listed workspaces
requested_by_upn_optional = ""      # Optional: requester email for audit

initialize_only = False               # Deployment use only; not exposed by the pipeline


In [ ]:
# PLATFORM SETUP — configured once by the scanner owner; regular members do not edit.

# Identity: user | spn_secret | spn_keyvault
auth_mode = "user"
spn_tenant_id = ""
spn_client_id = ""
spn_client_secret = ""              # Test only; do not save a real secret here
allow_plaintext_spn_secret = False

# Azure Key Vault settings for production SPN authentication
key_vault_name_or_uri = ""
kv_tenant_id_secret_name = ""       # Optional when spn_tenant_id is set above
kv_client_id_secret_name = ""       # Optional when spn_client_id is set above
kv_client_secret_name = ""          # Required for spn_keyvault

# Standard analysis profile
analysis_profile = "standard"       # standard | deep
run_bpa = True
bpa_extended = False
run_model_metadata_checks = True
run_vertipaq = True
vpa_read_stats_from_data = False
run_refresh_history = True
refresh_history_top_n = 20
run_unused_objects = False
unused_objects_method = "WorkspaceMonitoring"
workspace_monitoring_days = 14
run_direct_lake_checks = True
scan_profile = "workspace_user"     # workspace_user | governance_admin

# Finding thresholds
min_column_size_mb = 50.0
min_column_model_pct = 10.0
high_column_model_pct = 25.0
high_cardinality_threshold = 1_000_000
max_storage_findings_per_model = 30

# Output, authorization, and safeguards
output_schema = "smopt"             # Use "" for a non-schema Lakehouse
enforce_spn_workspace_access_precheck = True
default_authorized_viewer_upns = () # RLS grants are normally maintained separately
model_access_sync_mode = "none"
max_models_per_run = 50
max_retries = 2
retry_base_seconds = 5
include_default_semantic_models = False
excluded_model_names = ("ModelBPA", "Fabric Capacity Metrics")
fail_pipeline_if_any_model_fails = True
fail_pipeline_if_permission_precheck_fails = True
exit_notebook_with_summary = False

# Automatically generated per run; kept out of the member-facing parameter list.
scan_request_id = ""
requested_by_upn = (requested_by_upn_optional or "").strip()


In [ ]:
import hashlib
import json
import math
import re
import time
import traceback
import uuid
from contextlib import nullcontext
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version

import pandas as pd
import sempy
import sempy.fabric as fabric
import sempy_labs as labs
from delta.tables import DeltaTable
from pyspark.sql import types as T

try:
    from sempy_labs import directlake
except ImportError:
    directlake = None
try:
    from sempy_labs import semantic_model as labs_semantic_model
except ImportError:
    labs_semantic_model = None


def installed_version(distribution_name, module):
    try:
        return package_version(distribution_name)
    except PackageNotFoundError:
        return getattr(module, "__version__", "UNKNOWN")


SCANNER_VERSION = "2.6.1"
SOLUTION_STAGE = "OPPORTUNITY_DISCOVERY"
SCAN_PROFILE = str(scan_profile).strip().lower()
if SCAN_PROFILE not in {"workspace_user", "governance_admin"}:
    raise ValueError("scan_profile must be 'workspace_user' or 'governance_admin'.")
admin = None
if SCAN_PROFILE == "governance_admin":
    import sempy.fabric.admin as admin
SEMANTIC_LINK_VERSION = installed_version("semantic-link-sempy", sempy)
SEMANTIC_LINK_LABS_VERSION = installed_version("semantic-link-labs", labs)
set_service_principal = getattr(fabric, "set_service_principal", None)


def validate_runtime_capabilities():
    required = [
        (fabric, "resolve_workspace_name_and_id", "workspace resolution"),
        (fabric, "list_items", "semantic-model discovery"),
        (labs, "is_default_semantic_model", "default-model detection"),
    ]
    if run_bpa:
        required.append((labs, "run_model_bpa", "best-practice analysis"))
    if run_model_metadata_checks:
        required.append((labs, "get_semantic_model_bim", "semantic-model metadata inspection"))
    if run_vertipaq:
        required.append((labs, "vertipaq_analyzer", "VertiPaq analysis"))
    if run_refresh_history:
        required.append((fabric, "list_refresh_requests", "refresh history"))
    if run_unused_objects:
        required.append((labs_semantic_model, "find_unused_objects", "unused-object analysis"))
    if run_direct_lake_checks:
        required.append((directlake, "check_fallback_reason", "Direct Lake analysis"))
    if SCAN_PROFILE == "governance_admin":
        required.append((admin, "list_item_access_details", "item access snapshot"))
    if auth_mode.strip().lower() != "user":
        required.append((fabric, "set_service_principal", "service-principal authentication"))

    missing = [
        f"{getattr(owner, '__name__', 'module')}.{name} ({purpose})"
        for owner, name, purpose in required
        if owner is None or not callable(getattr(owner, name, None))
    ]
    if missing:
        raise RuntimeError(
            "Scanner runtime lacks APIs required by the selected run: "
            + "; ".join(missing)
            + f". Installed versions: semantic-link={SEMANTIC_LINK_VERSION}, "
            + f"semantic-link-labs={SEMANTIC_LINK_LABS_VERSION}."
        )


validate_runtime_capabilities()
print(
    f"Scanner {SCANNER_VERSION} ready; runtime capabilities validated | "
    f"semantic-link={SEMANTIC_LINK_VERSION} | "
    f"semantic-link-labs={SEMANTIC_LINK_LABS_VERSION} | profile={SCAN_PROFILE}"
)


In [ ]:
# ---------- Generic helpers ----------

UTC = timezone.utc
RUN_STARTED_AT = datetime.now(UTC)
SCAN_ID = str(uuid.uuid4())
REQUEST_ID = scan_request_id.strip() or SCAN_ID


def utcnow():
    return datetime.now(UTC)


def stable_id(*parts):
    payload = "|".join("" if p is None else str(p) for p in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def clean_string(value, max_len=8000):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    return str(value)[:max_len]


def json_safe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)


def json_dumps(value):
    return json.dumps(value, default=json_safe, ensure_ascii=False, sort_keys=True)


def canon(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")


def canonical_record(row):
    if hasattr(row, "to_dict"):
        row = row.to_dict()
    return {canon(k): v for k, v in dict(row).items()}


def pick(record, aliases, default=None):
    for alias in aliases:
        key = canon(alias)
        if key in record:
            value = record[key]
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                return value
    return default


def as_bool(value, default=False):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


_SIZE_UNITS = {"b": 1, "kb": 1024, "mb": 1024**2, "gb": 1024**3, "tb": 1024**4}


def as_number(value, default=None):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return default
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).replace(",", "").strip()
    match = re.fullmatch(r"(-?\d+(?:\.\d+)?)\s*([kmgt]?b)?", text, flags=re.I)
    if not match:
        return default
    number = float(match.group(1))
    unit = (match.group(2) or "").lower()
    return number * _SIZE_UNITS.get(unit, 1)


def as_int(value, default=None):
    number = as_number(value, default=None)
    return default if number is None else int(number)


def as_timestamp(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    ts = pd.to_datetime(value, utc=True, errors="coerce")
    return None if pd.isna(ts) else ts.to_pydatetime()


def duration_seconds(start_value, end_value):
    start = as_timestamp(start_value)
    end = as_timestamp(end_value)
    return None if start is None or end is None else max(0.0, (end - start).total_seconds())


def truncate_error(exc):
    return clean_string(f"{type(exc).__name__}: {exc}", max_len=4000)


def error_category(exc):
    text = str(exc).lower()
    if any(x in text for x in ["401", "unauthorized", "authentication"]):
        return "AUTHENTICATION"
    if any(x in text for x in ["403", "forbidden", "readwrite", "permission", "not authorized"]):
        return "AUTHORIZATION"
    if any(x in text for x in ["429", "throttl", "too many requests"]):
        return "THROTTLING"
    if any(x in text for x in ["timeout", "timed out"]):
        return "TIMEOUT"
    if any(x in text for x in ["not found", "doesn't exist", "does not exist"]):
        return "NOT_FOUND"
    return "ANALYSIS_ERROR"


def is_retryable(exc):
    category = error_category(exc)
    text = str(exc).lower()
    return category in {"THROTTLING", "TIMEOUT"} or any(x in text for x in ["500", "502", "503", "504"])


def with_retry(label, function):
    for attempt in range(max_retries + 1):
        try:
            return function()
        except Exception as exc:
            if attempt >= max_retries or not is_retryable(exc):
                raise
            wait_seconds = retry_base_seconds * (2**attempt)
            print(f"{label}: retry {attempt + 1}/{max_retries} in {wait_seconds}s ({error_category(exc)})")
            time.sleep(wait_seconds)


def validate_uuid(value, field_name):
    try:
        return str(uuid.UUID(str(value)))
    except Exception as exc:
        raise ValueError(f"{field_name} must be a GUID: {value}") from exc


In [ ]:
# ---------- Authentication, simple scope parsing, permission precheck, target resolution ----------

def authentication_context():
    mode = auth_mode.strip().lower()
    if mode == "user":
        return nullcontext()

    if mode == "spn_secret":
        if not allow_plaintext_spn_secret:
            raise ValueError("spn_secret is test-only. Set allow_plaintext_spn_secret=True explicitly.")
        if not all([spn_tenant_id, spn_client_id, spn_client_secret]):
            raise ValueError("spn_secret requires tenant ID, client ID and client secret.")
        return set_service_principal(
            tenant_id=spn_tenant_id,
            client_id=spn_client_id,
            client_secret=spn_client_secret,
        )

    if mode == "spn_keyvault":
        if not key_vault_name_or_uri or not kv_client_secret_name:
            raise ValueError("spn_keyvault requires key_vault_name_or_uri and kv_client_secret_name.")
        tenant_arg = (
            (key_vault_name_or_uri, kv_tenant_id_secret_name)
            if kv_tenant_id_secret_name
            else spn_tenant_id
        )
        client_arg = (
            (key_vault_name_or_uri, kv_client_id_secret_name)
            if kv_client_id_secret_name
            else spn_client_id
        )
        if not tenant_arg or not client_arg:
            raise ValueError("Provide tenant/client IDs directly or as Key Vault secret references.")
        return set_service_principal(
            tenant_id=tenant_arg,
            client_id=client_arg,
            client_secret=(key_vault_name_or_uri, kv_client_secret_name),
        )

    raise ValueError("auth_mode must be user, spn_secret or spn_keyvault.")


def parse_guid_list(value, field_name, required=False):
    """Accept comma, semicolon, whitespace, or newline separated GUIDs."""
    if value is None:
        raw_values = []
    elif isinstance(value, (list, tuple, set)):
        raw_values = [str(item).strip() for item in value]
    else:
        raw_values = re.split(r"[,;\s]+", str(value).strip())

    values = []
    seen = set()
    for raw_value in raw_values:
        if not raw_value:
            continue
        normalized = validate_uuid(raw_value, field_name).lower()
        if normalized not in seen:
            seen.add(normalized)
            values.append(normalized)
    if required and not values:
        raise ValueError(f"{field_name} is required. Enter at least one Workspace ID.")
    return values


def parse_scope():
    workspace_targets = parse_guid_list(workspace_ids, "workspace_ids", required=True)
    model_targets = parse_guid_list(model_ids_optional, "model_ids_optional", required=False)
    return workspace_targets, model_targets


def normalize_upns(values):
    if values is None:
        return []
    if isinstance(values, str):
        values = re.split(r"[,;\s]+", values.strip())
    return sorted({str(v).strip().lower() for v in values if str(v).strip()})


def item_columns(df):
    mapping = {canon(c): c for c in df.columns}
    id_col = next((mapping[k] for k in ["id", "item_id", "dataset_id", "semantic_model_id"] if k in mapping), None)
    name_col = next((mapping[k] for k in ["display_name", "name", "item_name", "dataset_name", "semantic_model_name"] if k in mapping), None)
    if not id_col or not name_col:
        raise ValueError(f"Unable to locate ID/name columns in list_items result: {list(df.columns)}")
    return id_col, name_col


WORKSPACE_PRECHECK_CACHE = {}


def precheck_workspace_access(workspace_id):
    workspace_id = validate_uuid(workspace_id, "workspace_id")
    cache_key = workspace_id.lower()
    if cache_key in WORKSPACE_PRECHECK_CACHE:
        return WORKSPACE_PRECHECK_CACHE[cache_key]

    mode = auth_mode.strip().lower()
    result = {
        "status": "NOT_CHECKED_USER_MODE" if mode == "user" else "NOT_ENFORCED",
        "workspace_name": None,
        "workspace_role": None,
        "message": "User-mode scans use the current identity and workspace-scoped APIs." if mode == "user" else "SPN effective-access precheck is disabled.",
    }
    if mode == "user" or not enforce_spn_workspace_access_precheck:
        WORKSPACE_PRECHECK_CACHE[cache_key] = result
        return result

    try:
        workspace_name, normalized_workspace_id = with_retry(
            "workspace_access_resolution",
            lambda: fabric.resolve_workspace_name_and_id(workspace_id),
        )
        with_retry(
            "workspace_semantic_model_access",
            lambda: fabric.list_items(item_type="SemanticModel", workspace=normalized_workspace_id),
        )
        result.update({
            "status": "PASSED",
            "workspace_name": str(workspace_name),
            "message": "Scanner identity can resolve the approved workspace and list its semantic models.",
        })
    except Exception as exc:
        result.update({
            "status": "FAILED_WORKSPACE_ACCESS",
            "message": f"Workspace-scoped effective-access precheck failed: {truncate_error(exc)}",
        })

    WORKSPACE_PRECHECK_CACHE[cache_key] = result
    return result


def resolve_targets():
    workspace_targets, requested_model_ids = parse_scope()
    excluded_names = {str(name).strip().lower() for name in excluded_model_names}
    default_viewers = normalize_upns(default_authorized_viewer_upns)
    inventory = {}

    def load_workspace(ws_id):
        ws_id = validate_uuid(ws_id, "workspace_id")
        precheck = precheck_workspace_access(ws_id)
        if (
            auth_mode.strip().lower() != "user"
            and enforce_spn_workspace_access_precheck
            and precheck["status"] != "PASSED"
        ):
            raise PermissionError(precheck["message"])
        ws_name, normalized_ws_id = fabric.resolve_workspace_name_and_id(ws_id)
        items = fabric.list_items(item_type="SemanticModel", workspace=normalized_ws_id)
        id_col, name_col = item_columns(items)
        model_map = {
            str(row[id_col]).lower(): str(row[name_col])
            for _, row in items.iterrows()
        }
        return str(normalized_ws_id), str(ws_name), model_map, precheck

    for workspace_id in workspace_targets:
        normalized_ws_id, ws_name, model_map, precheck = load_workspace(workspace_id)
        for model_id, model_name in model_map.items():
            inventory.setdefault(model_id, []).append({
                "workspace_id": normalized_ws_id,
                "workspace_name": ws_name,
                "model_id": model_id,
                "model_name": model_name,
                "authorized_viewer_upns": default_viewers,
                "scope_source": "MODEL" if requested_model_ids else "WORKSPACE",
                "permission_precheck_status": precheck["status"],
                "scanner_workspace_role": precheck["workspace_role"],
                "permission_precheck_message": precheck["message"],
            })

    if requested_model_ids:
        targets = []
        missing = []
        ambiguous = []
        for model_id in requested_model_ids:
            matches = inventory.get(model_id, [])
            if not matches:
                missing.append(model_id)
            elif len(matches) > 1:
                ambiguous.append(model_id)
            else:
                target = matches[0]
                if target["model_name"].strip().lower() in excluded_names:
                    raise ValueError(f"Model {model_id} is excluded by platform configuration.")
                targets.append(target)
        if missing:
            raise ValueError(
                "The following Model IDs were not found in the supplied workspaces: " + ", ".join(missing)
            )
        if ambiguous:
            raise ValueError("A Model ID matched more than one supplied workspace: " + ", ".join(ambiguous))
    else:
        targets = [
            matches[0]
            for matches in inventory.values()
            if matches and matches[0]["model_name"].strip().lower() not in excluded_names
        ]

    if not include_default_semantic_models:
        eligible = []
        for target in targets:
            can_open_model = target["permission_precheck_status"] in {
                "PASSED", "NOT_CHECKED_USER_MODE", "NOT_ENFORCED"
            }
            if not can_open_model:
                eligible.append(target)
                continue
            is_default = labs.is_default_semantic_model(
                dataset=target["model_name"],
                workspace=target["workspace_id"],
            )
            if not is_default:
                eligible.append(target)
        targets = eligible

    if len(targets) > max_models_per_run:
        raise ValueError(
            f"The scope contains {len(targets)} models, above the configured limit of {max_models_per_run}. "
            "Enter selected Model IDs or ask the platform owner to raise the safeguard."
        )
    if not targets:
        raise ValueError("No eligible semantic models were found in the supplied scope.")
    return targets


if not requested_by_upn.strip():
    print("Note: requester email is blank; the scan can run, but the audit field will be empty.")
if analysis_profile.strip().lower() == "deep":
    run_unused_objects = True
if not run_bpa and not run_vertipaq:
    raise ValueError("At least one core analysis (BPA or VertiPaq) must be enabled.")
if model_access_sync_mode not in {"none", "merge", "replace_scanner_managed"}:
    raise ValueError("Invalid model_access_sync_mode.")


In [ ]:
# ---------- Delta table contracts and idempotent writers ----------

RUN_SCHEMA = T.StructType([
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("request_id", T.StringType(), False),
    T.StructField("requested_by_upn", T.StringType()),
    T.StructField("auth_mode", T.StringType()),
    T.StructField("analysis_profile", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("target_count", T.IntegerType()),
    T.StructField("success_count", T.IntegerType()),
    T.StructField("partial_count", T.IntegerType()),
    T.StructField("failed_count", T.IntegerType()),
    T.StructField("skipped_count", T.IntegerType()),
    T.StructField("semantic_link_version", T.StringType()),
    T.StructField("semantic_link_labs_version", T.StringType()),
    T.StructField("parameters_hash", T.StringType()),
    T.StructField("error_category", T.StringType()),
    T.StructField("error_message", T.StringType()),
])

MODEL_SCAN_SCHEMA = T.StructType([
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_id", T.StringType(), False),
    T.StructField("model_name", T.StringType()),
    T.StructField("scope_source", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("permission_precheck_status", T.StringType()),
    T.StructField("scanner_workspace_role", T.StringType()),
    T.StructField("permission_precheck_message", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("model_size_bytes", T.LongType()),
    T.StructField("overall_status", T.StringType()),
    T.StructField("bpa_status", T.StringType()),
    T.StructField("vpa_status", T.StringType()),
    T.StructField("refresh_status", T.StringType()),
    T.StructField("usage_status", T.StringType()),
    T.StructField("direct_lake_status", T.StringType()),
    T.StructField("access_snapshot_status", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("error_json", T.StringType()),
])

FINDING_SCHEMA = T.StructType([
    T.StructField("finding_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("solution_stage", T.StringType()),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("model_name", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("rule_id", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("confidence", T.StringType()),
    T.StructField("impact_area", T.StringType()),
    T.StructField("object_type", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("object_name", T.StringType()),
    T.StructField("finding_text", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("evidence_json", T.StringType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("reclaimable_upper_bound_bytes", T.LongType()),
    T.StructField("cu_saving_status", T.StringType()),
    T.StructField("benefit_validation_status", T.StringType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("documentation_url", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

VPA_COLUMN_SCHEMA = T.StructType([
    T.StructField("evidence_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()),
    T.StructField("data_type", T.StringType()),
    T.StructField("encoding", T.StringType()),
    T.StructField("cardinality", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("model_size_pct", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

VPA_TABLE_SCHEMA = T.StructType([
    T.StructField("evidence_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("row_count", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("model_size_pct", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OBJECT_USAGE_SCHEMA = T.StructType([
    T.StructField("usage_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("object_type", T.StringType()),
    T.StructField("object_name", T.StringType()),
    T.StructField("is_used", T.BooleanType()),
    T.StructField("usage_count", T.LongType()),
    T.StructField("usage_method", T.StringType()),
    T.StructField("usage_window_days", T.IntegerType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

REFRESH_SCHEMA = T.StructType([
    T.StructField("refresh_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("request_id", T.StringType()),
    T.StructField("refresh_type", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("start_time", T.TimestampType()),
    T.StructField("end_time", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("captured_at", T.TimestampType()),
])

DIRECT_LAKE_SCHEMA = T.StructType([
    T.StructField("check_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("check_type", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("reason", T.StringType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

ACCESS_SNAPSHOT_SCHEMA = T.StructType([
    T.StructField("access_id", T.StringType(), False),
    T.StructField("scan_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("principal_id", T.StringType()),
    T.StructField("principal_name", T.StringType()),
    T.StructField("principal_type", T.StringType()),
    T.StructField("principal_upn", T.StringType()),
    T.StructField("permissions", T.StringType()),
    T.StructField("additional_permissions", T.StringType()),
    T.StructField("raw_json", T.StringType()),
    T.StructField("captured_at", T.TimestampType()),
])

MODEL_ACCESS_SCHEMA = T.StructType([
    T.StructField("access_key", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("model_id", T.StringType()),
    T.StructField("principal_upn", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("is_active", T.BooleanType()),
    T.StructField("valid_from", T.TimestampType()),
    T.StructField("valid_to", T.TimestampType()),
    T.StructField("updated_by", T.StringType()),
    T.StructField("updated_at", T.TimestampType()),
])

DIM_MODEL_SCHEMA = T.StructType([
    T.StructField("model_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("model_name", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("latest_scan_id", T.StringType()),
    T.StructField("last_scan_status", T.StringType()),
    T.StructField("last_scanned_at", T.TimestampType()),
])

TABLES = {
    "scan_run": (RUN_SCHEMA, ["scan_id"]),
    "model_scan": (MODEL_SCAN_SCHEMA, ["scan_id", "model_id"]),
    "finding": (FINDING_SCHEMA, ["finding_id"]),
    "vpa_column": (VPA_COLUMN_SCHEMA, ["evidence_id"]),
    "vpa_table": (VPA_TABLE_SCHEMA, ["evidence_id"]),
    "object_usage": (OBJECT_USAGE_SCHEMA, ["usage_id"]),
    "refresh": (REFRESH_SCHEMA, ["refresh_id"]),
    "direct_lake": (DIRECT_LAKE_SCHEMA, ["check_id"]),
    "item_access_snapshot": (ACCESS_SNAPSHOT_SCHEMA, ["access_id"]),
    "model_access": (MODEL_ACCESS_SCHEMA, ["access_key"]),
    "dim_model": (DIM_MODEL_SCHEMA, ["model_id"]),
}


def table_name(logical_name):
    physical = f"smopt_{logical_name}"
    return f"{output_schema}.{physical}" if output_schema else physical


def ensure_tables():
    if output_schema:
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", output_schema):
            raise ValueError("output_schema contains invalid characters.")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{output_schema}`")
    for logical_name, (schema, _) in TABLES.items():
        name = table_name(logical_name)
        if not spark.catalog.tableExists(name):
            spark.createDataFrame([], schema).write.format("delta").mode("errorifexists").saveAsTable(name)
            continue
        existing_columns = {field.name.lower() for field in spark.table(name).schema.fields}
        missing_fields = [field for field in schema.fields if field.name.lower() not in existing_columns]
        if missing_fields:
            additions = ", ".join(
                f"`{field.name}` {field.dataType.simpleString().upper()}"
                for field in missing_fields
            )
            spark.sql(f"ALTER TABLE {name} ADD COLUMNS ({additions})")


def upsert_rows(logical_name, rows):
    if not rows:
        return
    schema, keys = TABLES[logical_name]
    source = spark.createDataFrame(rows, schema=schema)
    name = table_name(logical_name)
    condition = " AND ".join(f"t.`{key}` = s.`{key}`" for key in keys)
    (
        DeltaTable.forName(spark, name)
        .alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def safe_parameters_hash():
    safe = {
        "workspace_ids": workspace_ids,
        "model_ids_optional": model_ids_optional,
        "analysis_profile": analysis_profile,
        "auth_mode": auth_mode,
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "scan_profile": SCAN_PROFILE,
        "enforce_spn_workspace_access_precheck": enforce_spn_workspace_access_precheck,
        "run_bpa": run_bpa,
        "bpa_extended": bpa_extended,
        "run_model_metadata_checks": run_model_metadata_checks,
        "run_vertipaq": run_vertipaq,
        "vpa_read_stats_from_data": vpa_read_stats_from_data,
        "run_refresh_history": run_refresh_history,
        "run_unused_objects": run_unused_objects,
        "unused_objects_method": unused_objects_method,
        "run_direct_lake_checks": run_direct_lake_checks,
        "thresholds": [min_column_size_mb, min_column_model_pct, high_column_model_pct, high_cardinality_threshold],
    }
    return stable_id(json_dumps(safe))


In [ ]:
# ---------- Analysis result normalizers ----------

def finding_base(target, source, rule_name, object_type=None, table_name_value=None, object_name=None):
    now = utcnow()
    rule_id = stable_id(source, rule_name)
    return {
        "finding_id": stable_id(SCAN_ID, target["model_id"], source, rule_name, table_name_value, object_name),
        "scan_id": SCAN_ID,
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "source": source,
        "category": None,
        "rule_id": rule_id,
        "rule_name": clean_string(rule_name, 1000),
        "severity": "INFO",
        "confidence": "MEDIUM",
        "impact_area": "MODEL_QUALITY",
        "object_type": clean_string(object_type, 200),
        "table_name": clean_string(table_name_value, 1000),
        "object_name": clean_string(object_name, 1000),
        "finding_text": None,
        "recommended_action": None,
        "technical_evidence": None,
        "evidence_json": None,
        "estimated_saving_bytes_low": None,
        "estimated_saving_bytes_high": None,
        "reclaimable_upper_bound_bytes": None,
        "cu_saving_status": "NOT_ESTIMATED_STAGE_2_REQUIRED",
        "benefit_validation_status": "NOT_STARTED",
        "change_risk": "MEDIUM",
        "validation_required": True,
        "documentation_url": None,
        "detected_at": now,
    }


def normalize_bpa(target, bpa_df):
    findings = []
    if bpa_df is None or bpa_df.empty:
        return findings
    for _, raw_row in bpa_df.iterrows():
        record = canonical_record(raw_row)
        rule_name = clean_string(pick(record, ["Rule Name", "Rule", "Name"], "BPA rule"), 1000)
        object_type = clean_string(pick(record, ["Scope", "Object Type", "ObjectType"]), 200)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        object_value = clean_string(pick(record, ["Object Name", "Object", "ObjectName"]), 1000)
        description = clean_string(pick(record, ["Description", "Finding", "Message"]), 8000)
        severity = str(pick(record, ["Severity"], "Warning")).strip().upper()
        result = finding_base(target, "BPA", rule_name, object_type, table_value, object_value)
        result.update({
            "category": clean_string(pick(record, ["Category"]), 500),
            "severity": severity if severity in {"INFO", "WARNING", "ERROR"} else "WARNING",
            "confidence": "HIGH",
            "impact_area": "PERFORMANCE" if str(pick(record, ["Category"], "")).lower() == "performance" else "MODEL_QUALITY",
            "finding_text": description or rule_name,
            "recommended_action": description,
            "technical_evidence": f"Deterministic BPA rule violation: {rule_name}",
            "evidence_json": json_dumps(raw_row.to_dict()),
            "change_risk": "MEDIUM",
            "documentation_url": clean_string(pick(record, ["URL", "Documentation URL"]), 2000),
        })
        findings.append(result)
    return findings


def normalize_vpa(target, vpa_dict, model_size_bytes):
    column_rows, table_rows, storage_findings = [], [], []
    now = utcnow()
    model_size = model_size_bytes or 0

    columns_df = next((df for key, df in vpa_dict.items() if canon(key) == "columns"), pd.DataFrame())
    for _, raw_row in columns_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        column_value = clean_string(pick(record, ["Column Name", "Column"]), 1000)
        total_size = as_int(pick(record, ["Total Size", "Total Size Bytes"]))
        data_size = as_int(pick(record, ["Data Size", "Data Size Bytes"]))
        dictionary_size = as_int(pick(record, ["Dictionary Size", "Dictionary Size Bytes"]))
        hierarchy_size = as_int(pick(record, ["Hierarchy Size", "Hierarchy Size Bytes"]))
        cardinality = as_int(pick(record, ["Cardinality", "Column Cardinality"]))
        pct = (100.0 * total_size / model_size) if total_size is not None and model_size > 0 else None
        evidence_id = stable_id(SCAN_ID, target["model_id"], "VPA_COLUMN", table_value, column_value)
        column_rows.append({
            "evidence_id": evidence_id,
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "column_name": column_value,
            "data_type": clean_string(pick(record, ["Data Type", "Type"]), 200),
            "encoding": clean_string(pick(record, ["Encoding", "Encoding Hint"]), 200),
            "cardinality": cardinality,
            "data_size_bytes": data_size,
            "dictionary_size_bytes": dictionary_size,
            "hierarchy_size_bytes": hierarchy_size,
            "total_size_bytes": total_size,
            "model_size_pct": pct,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": now,
        })

    tables_df = next((df for key, df in vpa_dict.items() if canon(key) in {"tables", "table"}), pd.DataFrame())
    for _, raw_row in tables_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        total_size = as_int(pick(record, ["Total Size", "Total Size Bytes"]))
        pct = (100.0 * total_size / model_size) if total_size is not None and model_size > 0 else None
        table_rows.append({
            "evidence_id": stable_id(SCAN_ID, target["model_id"], "VPA_TABLE", table_value),
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "row_count": as_int(pick(record, ["Row Count", "Rows"])),
            "data_size_bytes": as_int(pick(record, ["Data Size", "Data Size Bytes"])),
            "dictionary_size_bytes": as_int(pick(record, ["Dictionary Size", "Dictionary Size Bytes"])),
            "hierarchy_size_bytes": as_int(pick(record, ["Hierarchy Size", "Hierarchy Size Bytes"])),
            "total_size_bytes": total_size,
            "model_size_pct": pct,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": now,
        })

    candidates = []
    min_bytes = int(min_column_size_mb * 1024 * 1024)
    for row in column_rows:
        size = row["total_size_bytes"] or 0
        pct = row["model_size_pct"] or 0.0
        cardinality = row["cardinality"] or 0
        if (size >= min_bytes and pct >= min_column_model_pct) or (size >= min_bytes and cardinality >= high_cardinality_threshold):
            candidates.append(row)
    candidates.sort(key=lambda x: x["total_size_bytes"] or 0, reverse=True)

    for row in candidates[:max_storage_findings_per_model]:
        data_type = (row["data_type"] or "").lower()
        rule_name = "Large high-cardinality column" if (row["cardinality"] or 0) >= high_cardinality_threshold else "Large storage contributor"
        finding = finding_base(target, "VPA_HEURISTIC", rule_name, "Column", row["table_name"], row["column_name"])
        size_mb = (row["total_size_bytes"] or 0) / 1024 / 1024
        pct = row["model_size_pct"]
        evidence_text = f"Column size={size_mb:.2f} MB; model share={pct:.2f}%" if pct is not None else f"Column size={size_mb:.2f} MB"
        if row["cardinality"] is not None:
            evidence_text += f"; cardinality={row['cardinality']:,}"
        recommendations = ["Confirm the column is used by reports, measures, relationships, external XMLA clients, and exports before changing it."]
        if "date" in data_type and "time" in data_type:
            recommendations.append("If business precision permits, reduce DateTime precision or split Date and Time usage to reduce cardinality.")
        elif any(x in data_type for x in ["string", "text"]):
            recommendations.append("Review long/high-cardinality text in fact tables; consider removal, normalization, or a surrogate key where semantically valid.")
        else:
            recommendations.append("Review granularity, data type, unused values, and whether aggregation can satisfy the reporting requirement.")
        finding.update({
            "category": "Storage",
            "severity": "WARNING" if (pct or 0) < high_column_model_pct else "ERROR",
            "confidence": "MEDIUM",
            "impact_area": "MODEL_SIZE",
            "finding_text": "The column is a material contributor to model memory. This is a prioritization signal, not proof that the column is unnecessary.",
            "recommended_action": " ".join(recommendations),
            "technical_evidence": evidence_text,
            "evidence_json": row["raw_json"],
            "reclaimable_upper_bound_bytes": row["total_size_bytes"],
            "change_risk": "HIGH",
        })
        storage_findings.append(finding)

    return column_rows, table_rows, storage_findings


def detect_storage_mode(vpa_dict):
    partitions_df = next((df for key, df in vpa_dict.items() if canon(key) in {"partitions", "partition"}), pd.DataFrame())
    if partitions_df.empty:
        return "UNKNOWN"
    values = set()
    for _, raw_row in partitions_df.iterrows():
        record = canonical_record(raw_row)
        mode = pick(record, ["Mode", "Storage Mode", "Partition Mode"])
        if mode is not None:
            values.add(str(mode).strip())
    return ", ".join(sorted(values)) if values else "UNKNOWN"


def normalize_usage(target, usage_df, vpa_columns):
    rows, findings = [], []
    if usage_df is None or usage_df.empty:
        return rows, findings
    size_lookup = {
        ((row["table_name"] or "").lower(), (row["column_name"] or "").lower()): row["total_size_bytes"]
        for row in vpa_columns
    }
    for _, raw_row in usage_df.iterrows():
        record = canonical_record(raw_row)
        table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
        object_type = clean_string(pick(record, ["Object Type", "Type"]), 200)
        object_value = clean_string(pick(record, ["Object Name", "Object"]), 1000)
        is_used = as_bool(pick(record, ["IsUsed", "Is Used"]), default=True)
        usage_count = as_int(pick(record, ["UsageCount", "Usage Count"]), default=0)
        usage_row = {
            "usage_id": stable_id(SCAN_ID, target["model_id"], "USAGE", table_value, object_type, object_value),
            "scan_id": SCAN_ID,
            "workspace_id": target["workspace_id"],
            "model_id": target["model_id"],
            "table_name": table_value,
            "object_type": object_type,
            "object_name": object_value,
            "is_used": is_used,
            "usage_count": usage_count,
            "usage_method": unused_objects_method,
            "usage_window_days": workspace_monitoring_days if unused_objects_method == "WorkspaceMonitoring" else None,
            "raw_json": json_dumps(raw_row.to_dict()),
            "detected_at": utcnow(),
        }
        rows.append(usage_row)
        if not is_used:
            upper_bound = size_lookup.get(((table_value or "").lower(), (object_value or "").lower()))
            finding = finding_base(target, "UNUSED_OBJECT_ANALYSIS", "Unused semantic model object", object_type, table_value, object_value)
            evidence_scope = (
                f"No references in {workspace_monitoring_days} days of Workspace Monitoring queries."
                if unused_objects_method == "WorkspaceMonitoring"
                else "No references found in downstream reports available in PBIR format."
            )
            finding.update({
                "category": "Usage",
                "severity": "WARNING" if upper_bound and upper_bound >= min_column_size_mb * 1024 * 1024 else "INFO",
                "confidence": "MEDIUM",
                "impact_area": "MODEL_SIZE" if object_type and object_type.lower() in {"column", "table"} else "MAINTAINABILITY",
                "finding_text": "The object was not observed in the selected usage evidence scope.",
                "recommended_action": "Validate external tools, thin reports in other workspaces, Analyze in Excel, paginated reports, subscriptions, APIs, and future-use requirements before removal.",
                "technical_evidence": evidence_scope,
                "evidence_json": usage_row["raw_json"],
                "reclaimable_upper_bound_bytes": upper_bound,
                "change_risk": "HIGH",
            })
            findings.append(finding)
    return rows, findings


In [ ]:
# ---------- Per-model scanner ----------

def skipped_permission_result(target):
    started = utcnow()
    completed = utcnow()
    message = target.get("permission_precheck_message") or "SPN workspace permission precheck did not pass."
    gated_status = "SKIPPED_PERMISSION"
    model_row = {
        "scan_id": SCAN_ID,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "scope_source": target["scope_source"],
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "permission_precheck_status": target.get("permission_precheck_status"),
        "scanner_workspace_role": target.get("scanner_workspace_role"),
        "permission_precheck_message": message,
        "capacity_id": None,
        "capacity_name": None,
        "storage_mode": "UNKNOWN",
        "model_size_bytes": None,
        "overall_status": gated_status,
        "bpa_status": gated_status if run_bpa else "NOT_RUN",
        "vpa_status": gated_status if run_vertipaq else "NOT_RUN",
        "metadata_status": gated_status if run_model_metadata_checks else "NOT_RUN",
        "refresh_status": gated_status if run_refresh_history else "NOT_RUN",
        "usage_status": gated_status if run_unused_objects else "NOT_RUN",
        "direct_lake_status": gated_status if run_direct_lake_checks else "NOT_RUN",
        "access_snapshot_status": "NOT_RUN_PERMISSION_PRECHECK",
        "finding_count": 0,
        "started_at": started,
        "completed_at": completed,
        "duration_seconds": (completed - started).total_seconds(),
        "error_json": json_dumps({"permission_precheck": message}),
    }
    dim_model_row = {
        "model_id": target["model_id"],
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_name": target["model_name"],
        "capacity_id": None,
        "capacity_name": None,
        "storage_mode": "UNKNOWN",
        "latest_scan_id": SCAN_ID,
        "last_scan_status": gated_status,
        "last_scanned_at": completed,
    }
    return {
        "model_row": model_row,
        "dim_model_row": dim_model_row,
        "findings": [],
        "vpa_columns": [],
        "vpa_tables": [],
        "usage_rows": [],
        "refresh_rows": [],
        "direct_lake_rows": [],
        "access_rows": [],
    }


ANALYSIS_STATUS_KEYS = ("bpa", "vpa", "metadata", "refresh", "usage", "direct_lake")


def classify_model_status(statuses):
    core_statuses = [
        statuses[key]
        for key, enabled in (("bpa", run_bpa), ("vpa", run_vertipaq))
        if enabled
    ]
    if core_statuses and all(status == "FAILED" for status in core_statuses):
        return "FAILED"
    if any(statuses[key] == "FAILED" for key in ANALYSIS_STATUS_KEYS):
        return "PARTIAL"
    return "SUCCEEDED"


def scan_one_model(target):
    if (
        auth_mode.strip().lower() != "user"
        and enforce_spn_workspace_access_precheck
        and target.get("permission_precheck_status") != "PASSED"
    ):
        print(
            f"Skipping {target['workspace_name']} / {target['model_name']}: "
            f"{target.get('permission_precheck_status')} — {target.get('permission_precheck_message')}"
        )
        return skipped_permission_result(target)

    started = utcnow()
    statuses = {
        "bpa": "NOT_RUN",
        "vpa": "NOT_RUN",
        "metadata": "NOT_RUN",
        "refresh": "NOT_RUN",
        "usage": "NOT_RUN",
        "direct_lake": "NOT_RUN",
        "access_snapshot": "NOT_RUN",
    }
    analysis_errors = {}
    optional_enrichment_warnings = {}
    findings, vpa_columns, vpa_tables = [], [], []
    usage_rows, refresh_rows, direct_lake_rows, access_rows = [], [], [], []
    capacity_id = capacity_name = None
    storage_mode = "UNKNOWN"
    model_size_bytes = None

    print(f"Scanning {target['workspace_name']} / {target['model_name']} ({target['model_id']})")

    get_capacity_id = getattr(labs, "get_capacity_id", None)
    if callable(get_capacity_id):
        try:
            capacity_id = clean_string(get_capacity_id(workspace=target["workspace_id"]), 200)
        except Exception as exc:
            optional_enrichment_warnings["capacity_id"] = truncate_error(exc)
    else:
        optional_enrichment_warnings["capacity_id"] = "Runtime API unavailable."

    get_capacity_name = getattr(labs, "get_capacity_name", None)
    if callable(get_capacity_name):
        try:
            capacity_name = clean_string(get_capacity_name(workspace=target["workspace_id"]), 500)
        except Exception as exc:
            optional_enrichment_warnings["capacity_name"] = truncate_error(exc)
    else:
        optional_enrichment_warnings["capacity_name"] = "Runtime API unavailable."

    get_model_size = getattr(labs, "get_semantic_model_size", None)
    if callable(get_model_size):
        try:
            model_size_bytes = int(with_retry(
                "model_size",
                lambda: get_model_size(dataset=target["model_name"], workspace=target["workspace_id"]),
            ))
        except Exception as exc:
            optional_enrichment_warnings["model_size"] = truncate_error(exc)
    else:
        optional_enrichment_warnings["model_size"] = "Runtime API unavailable."

    if run_bpa:
        try:
            bpa_df = with_retry(
                "bpa",
                lambda: labs.run_model_bpa(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    return_dataframe=True,
                    export=False,
                    extended=bpa_extended,
                    check_dependencies=True,
                ),
            )
            findings.extend(normalize_bpa(target, bpa_df))
            statuses["bpa"] = "SUCCEEDED"
        except Exception as exc:
            statuses["bpa"] = "FAILED"
            analysis_errors["bpa"] = truncate_error(exc)

    vpa_dict = {}
    if run_vertipaq:
        try:
            vpa_dict = with_retry(
                "vertipaq",
                lambda: labs.vertipaq_analyzer(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    read_stats_from_data=vpa_read_stats_from_data,
                ),
            )
            vpa_columns, vpa_tables, storage_findings = normalize_vpa(target, vpa_dict, model_size_bytes)
            findings.extend(storage_findings)
            storage_mode = detect_storage_mode(vpa_dict)
            statuses["vpa"] = "SUCCEEDED"
        except Exception as exc:
            statuses["vpa"] = "FAILED"
            analysis_errors["vpa"] = truncate_error(exc)

    if run_model_metadata_checks:
        try:
            model_metadata_bim = with_retry(
                "model_metadata",
                lambda: labs.get_semantic_model_bim(
                    dataset=target["model_id"],
                    workspace=target["workspace_id"],
                ),
            )
            findings.extend(normalize_model_metadata(target, model_metadata_bim, vpa_columns, vpa_tables))
            statuses["metadata"] = "SUCCEEDED"
        except Exception as exc:
            statuses["metadata"] = "FAILED"
            analysis_errors["model_metadata"] = truncate_error(exc)
    else:
        statuses["metadata"] = "NOT_RUN"

    if run_refresh_history:
        try:
            refresh_df = with_retry(
                "refresh_history",
                lambda: fabric.list_refresh_requests(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    top_n=refresh_history_top_n,
                ),
            )
            for index, raw_row in refresh_df.iterrows():
                record = canonical_record(raw_row)
                upstream_request_id = clean_string(pick(record, ["Request Id", "Id", "Refresh Id"]), 500)
                start_time = as_timestamp(pick(record, ["Start Time", "StartTime", "Start Date Time"] ))
                end_time = as_timestamp(pick(record, ["End Time", "EndTime", "End Date Time"] ))
                refresh_rows.append({
                    "refresh_id": stable_id(SCAN_ID, target["model_id"], "REFRESH", upstream_request_id or index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "request_id": upstream_request_id,
                    "refresh_type": clean_string(pick(record, ["Refresh Type", "Type"]), 200),
                    "status": clean_string(pick(record, ["Status"]), 200),
                    "start_time": start_time,
                    "end_time": end_time,
                    "duration_seconds": duration_seconds(start_time, end_time),
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "captured_at": utcnow(),
                })
            statuses["refresh"] = "SUCCEEDED"
        except Exception as exc:
            statuses["refresh"] = "FAILED"
            analysis_errors["refresh"] = truncate_error(exc)

    if run_unused_objects:
        try:
            usage_df = with_retry(
                "unused_objects",
                lambda: labs_semantic_model.find_unused_objects(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                    method=unused_objects_method,
                    workspace_monitoring_days=workspace_monitoring_days,
                    visualize=False,
                ),
            )
            usage_rows, usage_findings = normalize_usage(target, usage_df, vpa_columns)
            findings.extend(usage_findings)
            statuses["usage"] = "SUCCEEDED"
        except Exception as exc:
            statuses["usage"] = "FAILED"
            analysis_errors["usage"] = truncate_error(exc)

    is_direct_lake = "directlake" in storage_mode.replace(" ", "").lower()
    if run_direct_lake_checks and is_direct_lake:
        try:
            fallback_df = with_retry(
                "direct_lake_fallback",
                lambda: directlake.check_fallback_reason(
                    dataset=target["model_name"],
                    workspace=target["workspace_id"],
                ),
            )
            for index, raw_row in fallback_df.iterrows():
                record = canonical_record(raw_row)
                table_value = clean_string(pick(record, ["Table Name", "Table"]), 1000)
                reason = clean_string(pick(record, ["Fallback Reason", "Reason"]), 8000)
                status_value = "ISSUE" if reason and reason.strip().lower() not in {"none", "n/a", "no fallback"} else "OK"
                direct_lake_rows.append({
                    "check_id": stable_id(SCAN_ID, target["model_id"], "DIRECT_LAKE_FALLBACK", table_value, index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "check_type": "FALLBACK_REASON",
                    "table_name": table_value,
                    "status": status_value,
                    "reason": reason,
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "detected_at": utcnow(),
                })
                if status_value == "ISSUE":
                    finding = finding_base(target, "DIRECT_LAKE", "Direct Lake fallback risk", "Table", table_value, table_value)
                    finding.update({
                        "category": "Direct Lake",
                        "severity": "WARNING",
                        "confidence": "HIGH",
                        "impact_area": "QUERY_CU",
                        "finding_text": reason,
                        "recommended_action": "Resolve the reported fallback condition and validate query behavior and CU before/after the change.",
                        "technical_evidence": reason,
                        "evidence_json": json_dumps(raw_row.to_dict()),
                        "change_risk": "MEDIUM",
                    })
                    findings.append(finding)
            statuses["direct_lake"] = "SUCCEEDED"
        except Exception as exc:
            statuses["direct_lake"] = "FAILED"
            analysis_errors["direct_lake"] = truncate_error(exc)
    elif run_direct_lake_checks:
        statuses["direct_lake"] = "NOT_APPLICABLE"

    if SCAN_PROFILE == "governance_admin":
        try:
            access_df = with_retry(
                "item_access_snapshot",
                lambda: admin.list_item_access_details(
                    item=target["model_id"],
                    item_type="SemanticModel",
                    workspace=target["workspace_id"],
                ),
            )
            for index, raw_row in access_df.iterrows():
                record = canonical_record(raw_row)
                principal_id = clean_string(pick(record, ["User Id", "Graph Id", "Identifier"]), 500)
                principal_upn = clean_string(pick(record, ["User Principal Name", "Email Address"]), 1000)
                access_rows.append({
                    "access_id": stable_id(SCAN_ID, target["model_id"], principal_id, principal_upn, index),
                    "scan_id": SCAN_ID,
                    "workspace_id": target["workspace_id"],
                    "model_id": target["model_id"],
                    "principal_id": principal_id,
                    "principal_name": clean_string(pick(record, ["User Name", "Name"]), 1000),
                    "principal_type": clean_string(pick(record, ["User Type", "Principal Type"]), 200),
                    "principal_upn": principal_upn.lower() if principal_upn else None,
                    "permissions": clean_string(pick(record, ["Permissions", "Dataset User Access Right"]), 2000),
                    "additional_permissions": clean_string(pick(record, ["Additional Permissions"]), 2000),
                    "raw_json": json_dumps(raw_row.to_dict()),
                    "captured_at": utcnow(),
                })
            statuses["access_snapshot"] = "SUCCEEDED"
        except Exception as exc:
            statuses["access_snapshot"] = "FAILED"
            optional_enrichment_warnings["access_snapshot"] = truncate_error(exc)
    else:
        statuses["access_snapshot"] = "NOT_APPLICABLE_WORKSPACE_USER_PROFILE"

    overall = classify_model_status(statuses)
    error_details = {}
    if analysis_errors:
        error_details["analysis_errors"] = analysis_errors
    if optional_enrichment_warnings:
        error_details["optional_enrichment_warnings"] = optional_enrichment_warnings

    completed = utcnow()
    model_row = {
        "scan_id": SCAN_ID,
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_id": target["model_id"],
        "model_name": target["model_name"],
        "scope_source": target["scope_source"],
        "scanner_version": SCANNER_VERSION,
        "solution_stage": SOLUTION_STAGE,
        "permission_precheck_status": target.get("permission_precheck_status"),
        "scanner_workspace_role": target.get("scanner_workspace_role"),
        "permission_precheck_message": target.get("permission_precheck_message"),
        "capacity_id": capacity_id,
        "capacity_name": capacity_name,
        "storage_mode": storage_mode,
        "model_size_bytes": model_size_bytes,
        "overall_status": overall,
        "bpa_status": statuses["bpa"],
        "vpa_status": statuses["vpa"],
        "metadata_status": statuses["metadata"],
        "refresh_status": statuses["refresh"],
        "usage_status": statuses["usage"],
        "direct_lake_status": statuses["direct_lake"],
        "access_snapshot_status": statuses["access_snapshot"],
        "finding_count": len(findings),
        "started_at": started,
        "completed_at": completed,
        "duration_seconds": (completed - started).total_seconds(),
        "error_json": json_dumps(error_details) if error_details else None,
    }
    dim_model_row = {
        "model_id": target["model_id"],
        "workspace_id": target["workspace_id"],
        "workspace_name": target["workspace_name"],
        "model_name": target["model_name"],
        "capacity_id": capacity_id,
        "capacity_name": capacity_name,
        "storage_mode": storage_mode,
        "latest_scan_id": SCAN_ID,
        "last_scan_status": overall,
        "last_scanned_at": completed,
    }
    return {
        "model_row": model_row,
        "dim_model_row": dim_model_row,
        "findings": findings,
        "vpa_columns": vpa_columns,
        "vpa_tables": vpa_tables,
        "usage_rows": usage_rows,
        "refresh_rows": refresh_rows,
        "direct_lake_rows": direct_lake_rows,
        "access_rows": access_rows,
    }


In [ ]:
# ---------- Explicit RLS grant synchronization ----------

def sync_explicit_model_access(targets):
    if model_access_sync_mode == "none":
        return
    eligible_targets = [
        target
        for target in targets
        if target.get("permission_precheck_status")
        in {"PASSED", "NOT_CHECKED_USER_MODE", "NOT_ENFORCED"}
    ]
    now = utcnow()
    requested_by = requested_by_upn.strip().lower() or None

    if model_access_sync_mode == "replace_scanner_managed":
        access_table = table_name("model_access")
        for target in eligible_targets:
            current_upns = target["authorized_viewer_upns"]
            predicate = (
                f"model_id = '{target['model_id']}' AND source = 'SCAN_REQUEST_EXPLICIT' AND is_active = true"
            )
            if current_upns:
                escaped = ",".join("'" + upn.replace("'", "''") + "'" for upn in current_upns)
                predicate += f" AND principal_upn NOT IN ({escaped})"
            DeltaTable.forName(spark, access_table).update(
                condition=predicate,
                set={"is_active": "false", "valid_to": "current_timestamp()", "updated_at": "current_timestamp()"},
            )

    rows = []
    for target in eligible_targets:
        for upn in target["authorized_viewer_upns"]:
            rows.append({
                "access_key": stable_id(target["model_id"], upn, "SCAN_REQUEST_EXPLICIT"),
                "workspace_id": target["workspace_id"],
                "model_id": target["model_id"],
                "principal_upn": upn,
                "source": "SCAN_REQUEST_EXPLICIT",
                "is_active": True,
                "valid_from": now,
                "valid_to": None,
                "updated_by": requested_by,
                "updated_at": now,
            })
    upsert_rows("model_access", rows)


In [ ]:
# ---------- V2 AI-friendly current-state consumption contract ----------

"""Deterministic quality grading for scanner findings and recommendations.

This dependency-free source is embedded into the Fabric scanner notebook by
``upgrade_notebook_v2.py`` and is also importable for local contract tests.
"""

ACTIONABLE = "ACTIONABLE"
REVIEW_REQUIRED = "REVIEW_REQUIRED"
INFORMATIONAL = "INFORMATIONAL"
SUPPRESSED = "SUPPRESSED"


# Only recommendations with a deterministic metadata operation belong in the
# approval-controlled script queue.  A broad domain such as "Formatting" is not
# sufficient: it also contains semantic decisions (data category, data type,
# business format selection) that cannot be generated safely without user input.
SCRIPT_CANDIDATE_RULE_FRAGMENTS = (
    "DO NOT SUMMARIZE NUMERIC COLUMNS",
    "HIDE FOREIGN KEYS",
    "MARK PRIMARY KEYS",
    "WHOLE NUMBERS SHOULD BE FORMATTED WITH THOUSANDS SEPARATORS AND NO DECIMALS",
    "FORMAT FLAG COLUMNS AS YES/NO VALUE STRINGS",
)

AUTO_DATE_ROOT_CAUSE_KEY = "AUTO_DATE_TIME"
AUTO_DATE_ROOT_CAUSE_SOURCE = "ROOT_CAUSE_CONSOLIDATION"
AUTO_DATE_ROOT_CAUSE_DOMAIN = "Date handling"
AUTO_DATE_ROOT_CAUSE_TITLE = "Replace Auto Date/Time with an explicit date dimension"
AUTO_DATE_ROOT_CAUSE_ACTION = (
    "Disable Auto Date/Time, create and mark an explicit date dimension, update relationships and calculations, "
    "then refresh and regression-test dependent reports."
)


def _normalized(value):
    return str(value or "").strip().upper()


def is_system_generated_date_object(finding):
    """Return True for Power BI Auto Date/Time implementation objects."""
    prefixes = ("LOCALDATETABLE_", "DATETABLETEMPLATE_")
    return any(
        _normalized(name).startswith(prefixes)
        for name in (finding.get("table_name"), finding.get("object_name"))
    )


def is_auto_date_root_cause_finding(finding):
    """Identify direct evidence that the model uses Auto Date/Time."""
    rule_name = _normalized(finding.get("rule_name"))
    return (
        is_system_generated_date_object(finding)
        or rule_name.startswith("MQ020:")
        or "AUTO DATE/TIME" in rule_name
    )


def is_date_table_companion_finding(finding):
    """Identify the missing-explicit-date-table finding that can share this root cause."""
    return _normalized(finding.get("rule_name")).startswith("MQ022:")


def root_cause_grouping(finding, auto_date_present):
    """Return a canonical rollup key while leaving the raw finding unchanged."""
    belongs_to_auto_date_root_cause = is_auto_date_root_cause_finding(finding) or (
        auto_date_present and is_date_table_companion_finding(finding)
    )
    if not belongs_to_auto_date_root_cause:
        return None
    return {
        "key": AUTO_DATE_ROOT_CAUSE_KEY,
        "source": AUTO_DATE_ROOT_CAUSE_SOURCE,
        "domain": AUTO_DATE_ROOT_CAUSE_DOMAIN,
        "title": AUTO_DATE_ROOT_CAUSE_TITLE,
        "action": AUTO_DATE_ROOT_CAUSE_ACTION,
    }


def _is_auto_date_root_cause_group(findings, title=None):
    """Confirm that a grouped recommendation/opportunity is one Auto Date root cause."""
    if not findings:
        return False
    if _normalized(title) == _normalized(AUTO_DATE_ROOT_CAUSE_TITLE):
        return True
    has_direct_evidence = any(is_auto_date_root_cause_finding(row) for row in findings)
    return has_direct_evidence and all(
        is_auto_date_root_cause_finding(row) or is_date_table_companion_finding(row)
        for row in findings
    )


def priority_band(score):
    """Map a stable 0-100 score to an explicit operational priority band."""
    if score >= 80:
        return "P1_CRITICAL"
    if score >= 65:
        return "P2_HIGH"
    if score >= 40:
        return "P3_MEDIUM"
    return "P4_LOW"


def _is_script_candidate(title):
    normalized = _normalized(title)
    return any(fragment in normalized for fragment in SCRIPT_CANDIDATE_RULE_FRAGMENTS)


def grade_finding(finding):
    """Grade one raw finding without discarding its original evidence."""
    description = str(finding.get("finding_text") or "").strip()
    evidence = str(finding.get("technical_evidence") or "").strip()
    action = str(finding.get("recommended_action") or "").strip()
    severity = _normalized(finding.get("severity"))
    confidence = _normalized(finding.get("confidence"))
    risk = _normalized(finding.get("change_risk"))

    suppression_reason = None
    if not description and not evidence:
        status = SUPPRESSED
        reason = "No description or technical evidence was supplied; retain for audit but exclude from the action queue."
        suppression_reason = reason
    elif is_system_generated_date_object(finding):
        status = SUPPRESSED
        reason = "System-generated Auto Date/Time object; remediate the model-level root cause instead of editing the generated object."
        suppression_reason = reason
    elif not evidence:
        status = REVIEW_REQUIRED
        reason = "A description is available, but technical evidence is missing; confirm the condition before implementation."
    elif not action:
        status = INFORMATIONAL
        reason = "Evidence is retained, but the scanner did not supply a concrete remediation action."
    elif severity in {"INFO", "LOW"}:
        status = INFORMATIONAL
        reason = "Low-severity evidence is useful context but does not belong in the immediate action queue."
    elif confidence in {"LOW", "UNKNOWN"}:
        status = REVIEW_REQUIRED
        reason = "The finding requires human confirmation because confidence is low or unavailable."
    elif risk == "HIGH":
        status = REVIEW_REQUIRED
        reason = "The proposed change has high implementation risk and requires design review before execution."
    else:
        status = ACTIONABLE
        reason = "The finding has evidence, a concrete action, sufficient confidence, and acceptable change risk."

    severity_points = {
        "CRITICAL": 70, "ERROR": 50, "HIGH": 45, "WARNING": 30,
        "MEDIUM": 25, "INFO": 10, "LOW": 5,
    }.get(severity, 0)
    # Treat absent confidence as neutral/medium rather than silently emptying
    # the actionable queue for collectors that do not publish this attribute.
    confidence_points = {"HIGH": 10, "MEDIUM": 5, "LOW": 0, "UNKNOWN": 0}.get(confidence, 5)
    risk_points = {"LOW": 5, "MEDIUM": 0, "HIGH": -10}.get(risk, 0)
    saving = max(
        int(finding.get("estimated_saving_bytes_low") or 0),
        int(finding.get("estimated_saving_bytes_high") or 0),
        int(finding.get("reclaimable_upper_bound_bytes") or 0),
    )
    score = severity_points + confidence_points + risk_points
    score += 5 if evidence else 0
    score += 10 if action else 0
    score += 5 if any(token in _normalized(finding.get("impact_area")) for token in (
        "PERFORMANCE", "MODEL_SIZE", "REFRESH", "QUERY", "CAPACITY",
    )) else 0
    score += 15 if saving > 0 else 0
    if status == SUPPRESSED:
        score = 0
    elif status == INFORMATIONAL:
        score = min(score, 39)
    elif severity != "CRITICAL" and saving <= 0:
        # P1 is reserved for explicitly critical evidence or quantified impact.
        # Finding volume is exposed separately and must not manufacture urgency.
        score = min(score, 79)
    score = max(0, min(100, score))

    return {
        "actionability_status": status,
        "actionability_reason": reason,
        "suppression_reason": suppression_reason,
        "finding_priority_score": score,
        "finding_priority_band": priority_band(score),
    }


def _why_it_matters(domain):
    text = _normalized(domain)
    if "DAX" in text or "EXPRESSION" in text:
        return "Improves calculation correctness, maintainability, and representative query performance."
    if any(token in text for token in ("PERFORMANCE", "STORAGE", "VERTIPAQ")):
        return "Reduces model size, refresh cost, memory pressure, and interactive query latency."
    if "FORMAT" in text:
        return "Improves semantic consistency and makes the model easier for users and AI agents to interpret."
    if any(token in text for token in ("MAINTENANCE", "GOVERNANCE", "BEST PRACTICE")):
        return "Reduces support cost and makes future model changes safer and easier to review."
    return "Addresses model quality or operational risk while preserving traceable evidence for validation."


def _validation_method(domain):
    text = _normalized(domain)
    if "DAX" in text or "EXPRESSION" in text:
        return "Compare representative query results and duration before and after the change, then rerun BPA."
    if any(token in text for token in ("PERFORMANCE", "STORAGE", "VERTIPAQ")):
        return "Compare model size, refresh duration, and representative query duration; rerun storage analysis."
    return "Rerun BPA and the scanner, then complete model refresh and report smoke tests."


def grade_recommendation(findings, domain, title, action):
    """Aggregate finding grades into an implementation-oriented recommendation."""
    grades = [grade_finding(row) for row in findings]
    statuses = [grade["actionability_status"] for grade in grades]
    auto_date_root_cause = _is_auto_date_root_cause_group(findings, title)

    if auto_date_root_cause:
        status = REVIEW_REQUIRED
        reason = (
            "BPA generated-date-object evidence and model-metadata date findings were consolidated into one "
            "model-level remediation that requires relationship and calculation review."
        )
        title = AUTO_DATE_ROOT_CAUSE_TITLE
        action = AUTO_DATE_ROOT_CAUSE_ACTION
        score = 72
    else:
        if ACTIONABLE in statuses:
            status = ACTIONABLE
            reason = "At least one linked finding meets the evidence, confidence, action, and risk thresholds for execution."
        elif REVIEW_REQUIRED in statuses:
            status = REVIEW_REQUIRED
            reason = "Linked findings require human confirmation or design review before implementation."
        elif INFORMATIONAL in statuses:
            status = INFORMATIONAL
            reason = "Linked findings are valid context but do not yet form an executable change."
        else:
            status = SUPPRESSED
            reason = "All linked findings are suppressed from the action queue while remaining available for audit."
        # Finding count is an impact/breadth dimension, not evidence of criticality.
        # Keep it in affected_finding_count instead of allowing volume to promote a
        # recommendation into a higher operational priority band.
        score = max((grade["finding_priority_score"] for grade in grades), default=0)

    highest_risk = max(
        (_normalized(row.get("change_risk")) for row in findings),
        key=lambda value: {"HIGH": 3, "MEDIUM": 2, "LOW": 1}.get(value, 0),
        default="",
    )
    if status in {INFORMATIONAL, SUPPRESSED}:
        automation = "NOT_ELIGIBLE"
    elif status == REVIEW_REQUIRED:
        automation = "MANUAL_REVIEW"
    elif highest_risk == "HIGH":
        automation = "MANUAL_ONLY"
    elif _is_script_candidate(title) and highest_risk in {"LOW", "MEDIUM"}:
        automation = "SCRIPT_CANDIDATE"
    else:
        automation = "MANUAL_REVIEW"

    return {
        "recommendation_title": title,
        "recommended_action": action,
        "actionability_status": status,
        "actionability_reason": reason,
        "recommendation_priority_score": score,
        "recommendation_priority_band": priority_band(score),
        "automation_eligibility": automation,
        "why_it_matters": _why_it_matters(domain),
        "validation_method": _validation_method(domain),
        "rollback_guidance": "Capture the original PBIP/TMDL state in source control and restore it if validation thresholds fail.",
        "actionable_finding_count": statuses.count(ACTIONABLE),
        "suppressed_finding_count": statuses.count(SUPPRESSED),
    }


def summarize_opportunity(findings, source, domain):
    """Build an explicit, AI-readable summary of an opportunity's finding mix."""
    grades = [grade_finding(row) for row in findings]
    counts = {
        ACTIONABLE: 0,
        REVIEW_REQUIRED: 0,
        INFORMATIONAL: 0,
        SUPPRESSED: 0,
    }
    for grade in grades:
        counts[grade["actionability_status"]] += 1
    return (
        f"{len(findings)} finding(s) from {source} in {domain}: "
        f"{counts[ACTIONABLE]} actionable, "
        f"{counts[REVIEW_REQUIRED]} review required, "
        f"{counts[INFORMATIONAL]} informational, and "
        f"{counts[SUPPRESSED]} suppressed."
    )


def grade_opportunity(findings):
    """Aggregate actionability and priority for an opportunity summary."""
    grades = [grade_finding(row) for row in findings]
    statuses = [grade["actionability_status"] for grade in grades]
    auto_date_root_cause = _is_auto_date_root_cause_group(findings)
    if auto_date_root_cause:
        status, score = REVIEW_REQUIRED, 72
    elif ACTIONABLE in statuses:
        status, score = ACTIONABLE, max(grade["finding_priority_score"] for grade in grades)
    elif REVIEW_REQUIRED in statuses:
        status, score = REVIEW_REQUIRED, max(grade["finding_priority_score"] for grade in grades)
    elif INFORMATIONAL in statuses:
        status, score = INFORMATIONAL, max(grade["finding_priority_score"] for grade in grades)
    else:
        status, score = SUPPRESSED, 0
    # Breadth remains available as actionable_finding_count and does not inflate
    # the opportunity's evidence-based priority.
    score = max(0, min(100, score))
    return {
        "actionability_status": status,
        "actionable_finding_count": statuses.count(ACTIONABLE),
        "review_required_finding_count": statuses.count(REVIEW_REQUIRED),
        "suppressed_finding_count": statuses.count(SUPPRESSED),
        "priority_score": score,
        "priority_band": priority_band(score),
    }


"""Deterministic semantic-model metadata heuristics.

The scanner feeds this module a standard Model.bim dictionary plus optional
VertiPaq column/table records.  The rules intentionally return a compact,
root-cause-oriented set of issues instead of one row for every generated or
missing-description object.
"""


import json
import re
from collections import defaultdict


SOURCE = "MODEL_METADATA_HEURISTIC"


def _text(value):
    return str(value or "").strip()


def _key(mapping, name, default=None):
    if not isinstance(mapping, dict):
        return default
    wanted = re.sub(r"[^a-z0-9]", "", name.lower())
    for key, value in mapping.items():
        if re.sub(r"[^a-z0-9]", "", str(key).lower()) == wanted:
            return value
    return default


def _list(mapping, name):
    value = _key(mapping, name, [])
    return value if isinstance(value, list) else []


def _model(bim):
    if not isinstance(bim, dict):
        return {}
    model = _key(bim, "model")
    if isinstance(model, dict):
        return model
    database = _key(bim, "database")
    nested = _key(database, "model") if isinstance(database, dict) else None
    return nested if isinstance(nested, dict) else bim


def _name(obj):
    return _text(_key(obj, "name"))


def _bool(obj, name, default=False):
    value = _key(obj, name, default)
    if isinstance(value, bool):
        return value
    return _text(value).lower() in {"true", "1", "yes"}


def _expression(obj):
    direct = _key(obj, "expression")
    if isinstance(direct, list):
        direct = "\n".join(map(str, direct))
    if direct is not None:
        return _text(direct)
    for partition in _list(obj, "partitions"):
        source = _key(partition, "source", {})
        value = _key(source, "expression") if isinstance(source, dict) else None
        if isinstance(value, list):
            value = "\n".join(map(str, value))
        if value:
            return _text(value)
    return ""


def _canon_expression(value):
    value = re.sub(r"//.*?$|/\*.*?\*/", "", _text(value), flags=re.M | re.S)
    return re.sub(r"\s+", "", value).lower()


def _column_ref_variants(table_name, column_name):
    """Return normalized DAX reference forms for one model column."""
    table_name = _text(table_name)
    column_name = _text(column_name)
    if not table_name or not column_name:
        return set()
    return {
        _canon_expression(f"{table_name}[{column_name}]"),
        _canon_expression(f"'{table_name}'[{column_name}]"),
    }


def _is_generated_table_name(table_name):
    return _text(table_name).lower().startswith(("localdatetable_", "datetabletemplate_"))


def _is_numeric_data_type(data_type):
    normalized = re.sub(r"[^a-z0-9]", "", _text(data_type).lower())
    return normalized in {
        "byte", "currency", "decimal", "decimal128", "double", "fixeddecimal",
        "float", "int", "int16", "int32", "int64", "integer", "long",
        "number", "real", "single", "uint16", "uint32", "uint64", "whole",
        "wholenumber",
    }


def _format_references_numeric_column(expression, columns):
    """Return True when FORMAT's value expression references a known numeric column."""
    numeric_names = {
        _name(column)
        for column in columns
        if _name(column) and _is_numeric_data_type(_key(column, "dataType"))
    }
    if not numeric_names:
        return False
    format_arguments = re.findall(r"\bFORMAT\s*\(\s*([^,\r\n]+)", _text(expression), re.I)
    for argument in format_arguments:
        for column_name in numeric_names:
            if re.search(rf"\[\s*{re.escape(column_name)}\s*\]", argument, re.I):
                return True
    return False


def _measure_is_text_like(measure, expression):
    """Identify measures whose result is intentionally text and needs no format string."""
    if "string" in _text(_key(measure, "dataType")).lower():
        return True
    if re.search(r"(?:label|name|title|text|description|message|caption)\s*$", _name(measure), re.I):
        return True
    canonical = _canon_expression(expression)
    return "format(" in canonical or "concatenate(" in canonical or "&" in canonical


def _relationship_is_invoked(relationship, measure_expressions):
    """Match USERELATIONSHIP to the specific inactive relationship it invokes."""
    from_refs = _column_ref_variants(
        _key(relationship, "fromTable"), _key(relationship, "fromColumn")
    )
    to_refs = _column_ref_variants(
        _key(relationship, "toTable"), _key(relationship, "toColumn")
    )
    for expression in measure_expressions:
        canonical = _canon_expression(expression)
        if "userelationship(" not in canonical:
            continue
        if any(ref in canonical for ref in from_refs) and any(ref in canonical for ref in to_refs):
            return True
    return False


def _issue(code, rule_name, category, severity, object_type, table_name, object_name,
           description, action, evidence, confidence="HIGH", risk="MEDIUM",
           impact="MODEL_QUALITY"):
    return {
        "rule_code": code,
        "source": SOURCE,
        "rule_name": rule_name,
        "category": category,
        "severity": severity,
        "confidence": confidence,
        "impact_area": impact,
        "object_type": object_type,
        "table_name": table_name,
        "object_name": object_name,
        "finding_text": description,
        "recommended_action": action,
        "technical_evidence": evidence,
        "evidence_json": json.dumps({"rule_code": code, "evidence": evidence}, ensure_ascii=False),
        "change_risk": risk,
    }


def analyze_model_bim(bim, vpa_columns=None, vpa_tables=None):
    """Return compact deterministic findings for one semantic model."""
    model = _model(bim)
    tables = _list(model, "tables")
    relationships = _list(model, "relationships")
    roles = _list(model, "roles")
    perspectives = _list(model, "perspectives")
    findings = []

    table_by_name = {_name(table): table for table in tables if _name(table)}
    relationship_tables = defaultdict(int)
    for rel in relationships:
        for field in ("fromTable", "toTable"):
            table_name = _text(_key(rel, field))
            if table_name:
                relationship_tables[table_name] += 1

    all_measure_expressions = []
    measure_records = []
    for table in tables:
        table_name = _name(table)
        for measure in _list(table, "measures"):
            expression = _expression(measure)
            all_measure_expressions.append(expression)
            measure_records.append((table_name, measure, expression))

    # Star-schema structure and table naming.
    technical_names = []
    prefix_names = []
    auto_date_names = []
    wide_root_cause_tables = set()
    for table in tables:
        table_name = _name(table)
        columns = _list(table, "columns")
        measures = _list(table, "measures")
        lower_name = table_name.lower()
        if re.match(r"^(stg|stage|temp|tmp|vw)[_ ]", lower_name) or re.match(r"^dim_.*\d+$", lower_name):
            technical_names.append(table_name)
        if re.match(r"^(fact|dim)[A-Z_]", table_name) or lower_name.startswith(("fact", "dim")):
            prefix_names.append(table_name)
        if _is_generated_table_name(table_name):
            auto_date_names.append(table_name)

        visible_string_columns = [
            column for column in columns
            if "string" in _text(_key(column, "dataType")).lower() and not _bool(column, "isHidden")
        ]
        if len(columns) >= 25 and len(visible_string_columns) >= 5 and relationship_tables[table_name] == 0:
            wide_root_cause_tables.add(table_name)
            findings.append(_issue(
                "MQ001", "Wide denormalized fact-grain table", "Model structure", "ERROR", "Table",
                table_name, table_name,
                "A wide disconnected table mixes many descriptive text attributes with fact-grain data.",
                "Restore a star schema: keep additive events in facts and descriptive attributes in related dimensions.",
                f"columns={len(columns)}; visible_string_columns={len(visible_string_columns)}; relationships=0",
                risk="HIGH", impact="PERFORMANCE",
            ))

        if relationship_tables[table_name] == 0 and not measures and 0 < len(columns) <= 5 and not _bool(table, "isHidden"):
            findings.append(_issue(
                "MQ003", "Disconnected table without measures", "Model structure", "WARNING", "Table",
                table_name, table_name,
                "The table has no model relationships and contains no measures.",
                "Confirm the table has an intentional disconnected-table use case; otherwise relate or remove it.",
                f"columns={len(columns)}; relationships=0; measures=0", risk="HIGH",
            ))

    if technical_names:
        findings.append(_issue(
            "MQ004", "Technical or temporary table names", "Naming", "WARNING", "Model", None, None,
            "Technical/staging names reduce business readability and AI discoverability.",
            "Rename published semantic objects with clear business terms and keep staging objects outside the model.",
            "tables=" + ", ".join(sorted(technical_names)), risk="LOW",
        ))
    if prefix_names:
        findings.append(_issue(
            "MQ026", "Fact/Dim table prefixes exposed to users", "Naming", "INFO", "Model", None, None,
            "Technical Fact/Dim prefixes are exposed in the business layer.",
            "Use concise business-facing table names while retaining technical lineage in descriptions.",
            "tables=" + ", ".join(sorted(prefix_names)), risk="MEDIUM",
        ))

    # Duplicate table definitions: exact column signatures or a bare calculated-table reference.
    signatures = defaultdict(list)
    duplicate_copy_tables = set()
    for table in tables:
        if _is_generated_table_name(_name(table)):
            continue
        names = tuple(sorted(_name(c).lower() for c in _list(table, "columns") if _name(c)))
        if len(names) >= 3:
            signatures[names].append(_name(table))
        expression = _canon_expression(_expression(table)).strip("'")
        if expression in {name.lower() for name in table_by_name if name.lower() != _name(table).lower()}:
            duplicate_copy_tables.add(_name(table))
            findings.append(_issue(
                "MQ002", "Duplicate calculated table", "Model structure", "ERROR", "Table", _name(table), _name(table),
                "The calculated table is a direct copy of another model table.",
                "Remove the duplicate and reuse the original dimension; validate dependencies before deletion.",
                f"expression={_expression(table)}", risk="HIGH", impact="MODEL_SIZE",
            ))
    for signature, names in signatures.items():
        if len(names) > 1:
            findings.append(_issue(
                "MQ002", "Duplicate table column signature", "Model structure", "ERROR", "Model", None, None,
                "Multiple tables expose the same non-trivial column set.",
                "Confirm whether the tables are true role-playing dimensions; otherwise consolidate duplicate copies.",
                f"tables={', '.join(sorted(names))}; shared_columns={len(signature)}", risk="HIGH", impact="MODEL_SIZE",
            ))

    duplicate_columns = defaultdict(list)
    missing_descriptions = defaultdict(list)
    double_columns = []
    invalid_summarize = []
    for table in tables:
        table_name = _name(table)
        columns = _list(table, "columns")
        table_is_generated = _is_generated_table_name(table_name)
        table_is_exposed = not _bool(table, "isHidden") and not table_is_generated
        if table_is_exposed and not _text(_key(table, "description")):
            missing_descriptions["tables"].append(table_name)
        for column in columns:
            column_name = _name(column)
            data_type = _text(_key(column, "dataType")).lower()
            expression = _expression(column)
            canon_expr = _canon_expression(expression)
            object_ref = f"{table_name}[{column_name}]"
            column_is_exposed = table_is_exposed and not _bool(column, "isHidden")
            if column_is_exposed and not _text(_key(column, "description")):
                missing_descriptions["columns"].append(object_ref)
            if (
                column_is_exposed
                and table_name not in wide_root_cause_tables
                and table_name not in duplicate_copy_tables
                and column_name
                and not re.search(r"(?:key|id)$", column_name, re.I)
            ):
                duplicate_columns[column_name.lower()].append(object_ref)

            if "string" in data_type and ("date" in column_name.lower() or ("format(" in canon_expr and "datevalue(" in canon_expr)):
                findings.append(_issue(
                    "MQ005", "Date stored or calculated as text", "Date handling", "ERROR", "Column",
                    table_name, column_name, "A date-like column uses text storage or FORMAT-based text output.",
                    "Use a Date/DateTime typed column and apply display formatting separately.",
                    f"data_type={data_type}; expression={expression}", risk="MEDIUM",
                ))
            if "related(" in canon_expr:
                findings.append(_issue(
                    "MQ006", "Dimension attribute copied into fact with RELATED", "Model structure", "ERROR", "Column",
                    table_name, column_name, "A calculated column copies a related attribute into another table.",
                    "Keep the attribute in its dimension and use the relationship/filter context.",
                    f"expression={expression}", risk="MEDIUM", impact="MODEL_SIZE",
                ))
            if table_name.lower().startswith("fact") and expression and re.search(r"[+*/-]", expression) and not re.search(r"RELATED|FORMAT|CONCATENATE|RAND", expression, re.I):
                findings.append(_issue(
                    "MQ007", "Row arithmetic implemented as a calculated fact column", "DAX", "ERROR", "Column",
                    table_name, column_name, "Row-level arithmetic is persisted in a fact calculated column.",
                    "Prefer a measure when the result is an aggregation and validate filter-context behavior.",
                    f"expression={expression}", risk="HIGH", impact="MODEL_SIZE",
                ))
            if re.search(r"\b(RAND|RANDBETWEEN|NOW|TODAY)\s*\(", expression, re.I):
                findings.append(_issue(
                    "MQ008", "Volatile or non-deterministic calculated column", "DAX", "WARNING", "Column",
                    table_name, column_name, "The expression uses a volatile/non-deterministic function.",
                    "Replace it with deterministic source data or a controlled refresh-time value.",
                    f"expression={expression}", risk="MEDIUM",
                ))
            if "concatenate(" in canon_expr or canon_expr.count("&") >= 2:
                findings.append(_issue(
                    "MQ010", "Multi-attribute concatenated column", "Model structure", "WARNING", "Column",
                    table_name, column_name, "The column combines multiple independent attributes into one text value.",
                    "Keep attributes separate for filtering/grouping; add a display label only when required.",
                    f"expression={expression}", risk="MEDIUM",
                ))
            if (
                column_is_exposed
                and "format(" in canon_expr
                and "string" in data_type
                and "date" not in column_name.lower()
                and _format_references_numeric_column(expression, columns)
            ):
                findings.append(_issue(
                    "MQ011", "Numeric value formatted into a text column", "Data types", "WARNING", "Column",
                    table_name, column_name, "FORMAT converts a numeric value into text, preventing correct aggregation and sorting.",
                    "Keep the column numeric and use format metadata for presentation.",
                    f"data_type={data_type}; expression={expression}", risk="MEDIUM",
                ))
            if re.search(r"^(column\d+|zz_|junk|placeholder)", column_name, re.I):
                findings.append(_issue(
                    "MQ012", "Meaningless or junk column name", "Naming", "INFO", "Column",
                    table_name, column_name, "The column name signals a placeholder, generated field, or unused artifact.",
                    "Confirm usage, then rename with business meaning or remove it after dependency validation.",
                    f"column={object_ref}", risk="HIGH",
                ))
            if re.search(r"month.*name|name.*month", column_name, re.I) and "string" in data_type and not _text(_key(column, "sortByColumn")):
                findings.append(_issue(
                    "MQ025", "Month-name column without chronological sort", "Date handling", "INFO", "Column",
                    table_name, column_name, "A text month attribute has no sort-by column and can sort alphabetically.",
                    "Set Sort by column to the numeric month sequence.",
                    f"sortByColumn={_key(column, 'sortByColumn')}", risk="LOW",
                ))
            if re.fullmatch(r"\[?[^\[\]]+\]?", _text(expression)) and expression and column_name.lower() not in expression.lower():
                findings.append(_issue(
                    "MQ013", "Redundant calculated column alias", "Maintainability", "INFO", "Column",
                    table_name, column_name, "The calculated column directly aliases another column.",
                    "Reuse the source column or rename it at the semantic layer instead of persisting a clone.",
                    f"expression={expression}", risk="MEDIUM", impact="MODEL_SIZE",
                ))
            if data_type in {"double", "real"}:
                double_columns.append(object_ref)
            summarize = _text(_key(column, "summarizeBy")).lower()
            if summarize not in {"", "none", "donotsummarize"} and re.search(r"key|id|numberof|linenumber|year", column_name, re.I):
                invalid_summarize.append(f"{object_ref}={summarize}")

    for column_name, refs in duplicate_columns.items():
        if len({ref.split("[")[0] for ref in refs}) > 1:
            findings.append(_issue(
                "MQ009", "Ambiguous column name across tables", "Naming", "WARNING", "Model", None, column_name,
                "The same non-key column name is exposed by multiple tables.",
                "Use specific business names and descriptions so users and AI can distinguish the fields.",
                "objects=" + ", ".join(sorted(refs)), risk="LOW",
            ))
    if double_columns:
        findings.append(_issue(
            "MQ023", "Floating-point columns", "Data types", "WARNING", "Model", None, None,
            "Double/Real columns can introduce rounding ambiguity and weaker compression.",
            "Use fixed decimal, whole number, or scaled integer types where business precision permits.",
            "columns=" + ", ".join(sorted(double_columns)), risk="HIGH", impact="MODEL_SIZE",
        ))
    if invalid_summarize:
        findings.append(_issue(
            "MQ024", "Implicit aggregation enabled on identifiers", "Usability", "INFO", "Model", None, None,
            "Identifier/date-sequence columns allow implicit aggregation.",
            "Set Summarize by to None/Do not summarize for non-additive attributes.",
            "columns=" + ", ".join(sorted(invalid_summarize)), risk="LOW",
        ))

    # Measure expression rules.
    expression_groups = defaultdict(list)
    for table_name, measure, expression in measure_records:
        measure_name = _name(measure)
        canon_expr = _canon_expression(expression)
        object_ref = f"{table_name}[{measure_name}]"
        if canon_expr:
            expression_groups[canon_expr].append(object_ref)
        if not _bool(measure, "isHidden") and not _text(_key(measure, "description")):
            missing_descriptions["measures"].append(object_ref)
        if table_name.lower().startswith("dim") and re.search(r"\b(SUM|SUMX|COUNT|COUNTROWS|AVERAGE)\s*\(\s*Fact", expression, re.I):
            findings.append(_issue(
                "MQ014", "Fact aggregation measure stored in a dimension", "Measure organization", "ERROR", "Measure",
                table_name, measure_name, "A measure in a dimension table aggregates a fact-table value.",
                "Move the measure to a dedicated measure table or the relevant business subject area.",
                f"expression={expression}", risk="MEDIUM",
            ))
        if re.fullmatch(r"\s*\[[^\]]+\]\s*", expression or ""):
            findings.append(_issue(
                "MQ016", "Pass-through measure alias", "DAX", "INFO", "Measure",
                table_name, measure_name, "The measure is only a direct reference to another measure.",
                "Remove the alias or give it distinct business logic and documentation.",
                f"expression={expression}", risk="MEDIUM",
            ))
        if re.search(r"CALCULATE|FILTER", expression, re.I) and re.search(r"(?:=|<>|>=|<=|>|<)\s*(?:\d{4}|\"[^\"]+\")", expression):
            findings.append(_issue(
                "MQ017", "Hardcoded filter literal in measure", "DAX", "ERROR", "Measure",
                table_name, measure_name, "The measure embeds a fixed filter literal that can silently age or return blank results.",
                "Use model attributes, parameters, or relative logic and regression-test across the supported data range.",
                f"expression={expression}", risk="HIGH",
            ))
        if re.search(r"FILTER\s*\(\s*ALL(?:EXCEPT)?\s*\(", expression, re.I):
            findings.append(_issue(
                "MQ018", "FILTER over ALL table scan", "DAX performance", "ERROR", "Measure",
                table_name, measure_name, "FILTER(ALL(...)) can force unnecessary full-table iteration.",
                "Rewrite with a direct Boolean filter or narrower filter-removal semantics where equivalent.",
                f"expression={expression}", risk="HIGH", impact="PERFORMANCE",
            ))
        if (
            not _bool(measure, "isHidden")
            and not _text(_key(measure, "formatString"))
            and not _measure_is_text_like(measure, expression)
        ):
            findings.append(_issue(
                "MQ019", "Visible measure without format string", "Formatting", "WARNING", "Measure",
                table_name, measure_name, "A visible measure has no explicit format string.",
                "Apply a business-appropriate numeric, currency, percentage, or date format.",
                f"measure={object_ref}", risk="LOW",
            ))
    for expression, refs in expression_groups.items():
        if len(refs) > 1:
            findings.append(_issue(
                "MQ015", "Duplicate measure expression", "DAX", "ERROR", "Model", None, None,
                "Multiple measures have the same normalized DAX expression.",
                "Consolidate the measures and preserve aliases only when they carry documented business semantics.",
                "measures=" + ", ".join(sorted(refs)), risk="HIGH",
            ))

    if auto_date_names:
        findings.append(_issue(
            "MQ020", "Auto Date/Time tables present", "Date handling", "ERROR", "Model", None, None,
            "System-generated local date tables coexist with the published semantic model.",
            "Disable Auto Date/Time and migrate calculations/relationships to a marked explicit date dimension.",
            "tables=" + ", ".join(sorted(auto_date_names)), risk="HIGH", impact="MODEL_SIZE",
        ))
    hidden_related = sorted(
        name for name, table in table_by_name.items()
        if _bool(table, "isHidden") and relationship_tables[name] > 0
    )
    if hidden_related:
        findings.append(_issue(
            "MQ021", "Hidden tables participate in relationships", "Maintainability", "WARNING", "Model", None, None,
            "Hidden tables remain active in model relationships and obscure lineage.",
            "Document the role or replace generated/technical tables with an explicit maintained design.",
            "tables=" + ", ".join(hidden_related), risk="HIGH",
        ))
    date_candidates = [
        table for table in tables
        if "date" in _name(table).lower() or any("date" in _name(c).lower() for c in _list(table, "columns"))
    ]
    marked_dates = [
        _name(table) for table in tables
        if _bool(table, "isDateTable") or _text(_key(table, "dataCategory")).lower() == "time"
    ]
    if date_candidates and not marked_dates:
        findings.append(_issue(
            "MQ022", "No explicit table marked as the date table", "Date handling", "ERROR", "Model", None, None,
            "Date-like tables exist but none is explicitly marked as the model date table.",
            "Mark the conformed date dimension and validate time-intelligence calculations.",
            "date_candidates=" + ", ".join(sorted(_name(t) for t in date_candidates)), risk="HIGH",
        ))

    # Storage-aware high-cardinality text rule.
    for row in vpa_columns or []:
        table_name = _text(row.get("table_name"))
        column_name = _text(row.get("column_name"))
        data_type = _text(row.get("data_type")).lower()
        cardinality = int(row.get("cardinality") or 0)
        if table_name.lower().startswith("fact") and any(x in data_type for x in ("string", "text")) and cardinality >= 10000:
            findings.append(_issue(
                "MQ027", "High-cardinality text in a fact table", "Storage", "WARNING", "Column",
                table_name, column_name, "A fact-grain text column has high cardinality and can create a large dictionary.",
                "Validate report/export usage, then remove, normalize, or move it to an appropriate degenerate dimension.",
                f"cardinality={cardinality}; total_size_bytes={row.get('total_size_bytes')}", risk="HIGH", impact="MODEL_SIZE",
            ))

    description_counts = {kind: len(values) for kind, values in missing_descriptions.items() if values}
    if description_counts:
        sample = []
        for kind in sorted(missing_descriptions):
            sample.extend(missing_descriptions[kind][:5])
        findings.append(_issue(
            "MQ028", "Visible semantic objects lack descriptions", "AI readiness", "INFO", "Model", None, None,
            "Visible tables, columns, or measures lack business descriptions.",
            "Add concise business meaning, grain, calculation intent, units, and important caveats; prioritize user-facing objects.",
            f"counts={json.dumps(description_counts, sort_keys=True)}; sample={', '.join(sample)}", risk="LOW",
        ))

    implicit = _key(model, "discourageImplicitMeasures", False)
    if not _bool({"value": implicit}, "value") or not roles or not perspectives:
        findings.append(_issue(
            "MQ029", "Model governance features are incomplete", "Governance", "WARNING", "Model", None, None,
            "The model permits implicit measures and/or has no roles or perspectives.",
            "Review explicit-measure policy, data sensitivity, RLS requirements, and audience-specific perspectives.",
            f"discourageImplicitMeasures={implicit}; roles={len(roles)}; perspectives={len(perspectives)}", risk="HIGH",
        ))

    unresolved_relationships = defaultdict(list)
    for relationship in relationships:
        if _bool(relationship, "isActive", True):
            continue
        from_table = _text(_key(relationship, "fromTable"))
        from_column = _text(_key(relationship, "fromColumn"))
        to_table = _text(_key(relationship, "toTable"))
        to_column = _text(_key(relationship, "toColumn"))
        if not _relationship_is_invoked(relationship, all_measure_expressions):
            unresolved_relationships[(from_table, to_table)].append(
                f"{from_table}[{from_column}] -> {to_table}[{to_column}]"
            )
    for (from_table, to_table), relationship_refs in sorted(unresolved_relationships.items()):
        findings.append(_issue(
            "MQ030", "Inactive relationships without USERELATIONSHIP measures", "Relationships", "INFO", "Relationship group",
            from_table, f"{from_table} -> {to_table}",
            "One or more inactive relationships between the same table pair are not invoked by a matching USERELATIONSHIP measure.",
            "Confirm each role-playing relationship is required and add explicit measures or remove incomplete design artifacts.",
            f"count={len(relationship_refs)}; relationships=" + "; ".join(sorted(relationship_refs)), risk="HIGH",
        ))

    # Deterministic de-duplication protects idempotent output and opportunity counts.
    unique = {}
    for finding in findings:
        key = (
            finding["rule_code"], finding.get("table_name"), finding.get("object_name"),
            finding.get("technical_evidence"),
        )
        unique[key] = finding
    return list(unique.values())



def normalize_model_metadata(target, model_bim, vpa_columns, vpa_tables):
    findings = []
    for issue in analyze_model_bim(model_bim, vpa_columns, vpa_tables):
        rule_name = f"{issue['rule_code']}: {issue['rule_name']}"
        finding = finding_base(
            target,
            issue.get("source") or "MODEL_METADATA_HEURISTIC",
            rule_name,
            issue.get("object_type"),
            issue.get("table_name"),
            issue.get("object_name"),
        )
        finding.update({
            "category": issue.get("category"),
            "severity": issue.get("severity") or "INFO",
            "confidence": issue.get("confidence") or "HIGH",
            "impact_area": issue.get("impact_area") or "MODEL_QUALITY",
            "finding_text": issue.get("finding_text"),
            "recommended_action": issue.get("recommended_action"),
            "technical_evidence": issue.get("technical_evidence"),
            "evidence_json": issue.get("evidence_json"),
            "change_risk": issue.get("change_risk") or "MEDIUM",
        })
        findings.append(finding)
    return findings

BUSINESS_SCHEMAS = (
    "analysis_control",
    "semantic_model_metadata",
    "semantic_model_vertipaq",
    "semantic_model_best_practice",
    "semantic_model_optimization",
)

OPTIMIZATION_OVERVIEW_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType()),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("analysis_status", T.StringType()),
    T.StructField("analysis_completed_at", T.TimestampType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("semantic_model_size_bytes", T.LongType()),
    T.StructField("optimization_opportunity_count", T.IntegerType()),
    T.StructField("optimization_recommendation_count", T.IntegerType()),
    T.StructField("optimization_finding_count", T.IntegerType()),
    T.StructField("high_severity_finding_count", T.IntegerType()),
    T.StructField("actionable_recommendation_count", T.IntegerType()),
    T.StructField("review_required_recommendation_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("best_practice_analysis_status", T.StringType()),
    T.StructField("storage_analysis_status", T.StringType()),
    T.StructField("refresh_history_status", T.StringType()),
    T.StructField("refresh_history_record_count", T.IntegerType()),
    T.StructField("object_usage_analysis_status", T.StringType()),
    T.StructField("object_usage_observation_count", T.IntegerType()),
    T.StructField("direct_lake_analysis_status", T.StringType()),
    T.StructField("direct_lake_observation_count", T.IntegerType()),
    T.StructField("item_access_snapshot_status", T.StringType()),
    T.StructField("item_access_record_count", T.IntegerType()),
    T.StructField("data_availability_explanation", T.StringType()),
])

OPTIMIZATION_OPPORTUNITY_SCHEMA = T.StructType([
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("opportunity_title", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("highest_severity", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("recommendation_count", T.IntegerType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("opportunity_summary", T.StringType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionable_finding_count", T.IntegerType()),
    T.StructField("review_required_finding_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("priority_score", T.IntegerType()),
    T.StructField("priority_band", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_RECOMMENDATION_SCHEMA = T.StructType([
    T.StructField("recommendation_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("recommendation_title", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("affected_finding_count", T.IntegerType()),
    T.StructField("actionable_finding_count", T.IntegerType()),
    T.StructField("suppressed_finding_count", T.IntegerType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionability_reason", T.StringType()),
    T.StructField("recommendation_priority_score", T.IntegerType()),
    T.StructField("recommendation_priority_band", T.StringType()),
    T.StructField("automation_eligibility", T.StringType()),
    T.StructField("why_it_matters", T.StringType()),
    T.StructField("validation_method", T.StringType()),
    T.StructField("rollback_guidance", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_FINDING_SCHEMA = T.StructType([
    T.StructField("finding_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("finding_source", T.StringType()),
    T.StructField("optimization_domain", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("confidence", T.StringType()),
    T.StructField("impact_area", T.StringType()),
    T.StructField("affected_object_type", T.StringType()),
    T.StructField("affected_table_name", T.StringType()),
    T.StructField("affected_object_name", T.StringType()),
    T.StructField("object_scope", T.StringType()),
    T.StructField("display_table_name", T.StringType()),
    T.StructField("display_object_name", T.StringType()),
    T.StructField("finding_description", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("estimated_saving_bytes_low", T.LongType()),
    T.StructField("estimated_saving_bytes_high", T.LongType()),
    T.StructField("change_risk", T.StringType()),
    T.StructField("validation_required", T.BooleanType()),
    T.StructField("actionability_status", T.StringType()),
    T.StructField("actionability_reason", T.StringType()),
    T.StructField("suppression_reason", T.StringType()),
    T.StructField("finding_priority_score", T.IntegerType()),
    T.StructField("finding_priority_band", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

OPTIMIZATION_LINK_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("opportunity_id", T.StringType(), False),
    T.StructField("related_entity_id", T.StringType(), False),
])

COLUMN_STORAGE_SCHEMA = T.StructType([
    T.StructField("column_storage_record_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("column_name", T.StringType()),
    T.StructField("data_type", T.StringType()),
    T.StructField("encoding", T.StringType()),
    T.StructField("cardinality", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("percentage_of_semantic_model_size", T.DoubleType()),
    T.StructField("detected_at", T.TimestampType()),
])

ANALYSIS_RUN_SCHEMA = T.StructType([
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("scanner_version", T.StringType()),
    T.StructField("analysis_profile", T.StringType()),
    T.StructField("analysis_status", T.StringType()),
    T.StructField("permission_precheck_status", T.StringType()),
    T.StructField("best_practice_analysis_status", T.StringType()),
    T.StructField("storage_analysis_status", T.StringType()),
    T.StructField("refresh_history_status", T.StringType()),
    T.StructField("object_usage_analysis_status", T.StringType()),
    T.StructField("direct_lake_analysis_status", T.StringType()),
    T.StructField("item_access_snapshot_status", T.StringType()),
    T.StructField("finding_count", T.IntegerType()),
    T.StructField("started_at", T.TimestampType()),
    T.StructField("completed_at", T.TimestampType()),
    T.StructField("duration_seconds", T.DoubleType()),
    T.StructField("error_details", T.StringType()),
])

SEMANTIC_MODEL_SCHEMA = T.StructType([
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("workspace_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("capacity_id", T.StringType()),
    T.StructField("capacity_name", T.StringType()),
    T.StructField("storage_mode", T.StringType()),
    T.StructField("semantic_model_size_bytes", T.LongType()),
    T.StructField("latest_analysis_id", T.StringType()),
    T.StructField("latest_analysis_status", T.StringType()),
    T.StructField("latest_analysis_at", T.TimestampType()),
    T.StructField("scanner_version", T.StringType()),
])

BEST_PRACTICE_FINDING_SCHEMA = T.StructType([
    T.StructField("best_practice_finding_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("rule_id", T.StringType()),
    T.StructField("rule_name", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("affected_object_type", T.StringType()),
    T.StructField("affected_table_name", T.StringType()),
    T.StructField("affected_object_name", T.StringType()),
    T.StructField("finding_description", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("technical_evidence", T.StringType()),
    T.StructField("documentation_url", T.StringType()),
    T.StructField("detected_at", T.TimestampType()),
])

TABLE_STORAGE_SCHEMA = T.StructType([
    T.StructField("table_storage_record_id", T.StringType(), False),
    T.StructField("analysis_id", T.StringType(), False),
    T.StructField("workspace_name", T.StringType()),
    T.StructField("semantic_model_id", T.StringType(), False),
    T.StructField("semantic_model_name", T.StringType()),
    T.StructField("table_name", T.StringType()),
    T.StructField("row_count", T.LongType()),
    T.StructField("data_size_bytes", T.LongType()),
    T.StructField("dictionary_size_bytes", T.LongType()),
    T.StructField("hierarchy_size_bytes", T.LongType()),
    T.StructField("total_size_bytes", T.LongType()),
    T.StructField("percentage_of_semantic_model_size", T.DoubleType()),
    T.StructField("detected_at", T.TimestampType()),
])

CURATED_TABLES = {
    "analysis_runs": ("analysis_control", "semantic_model_analysis_runs", ANALYSIS_RUN_SCHEMA),
    "semantic_models": ("semantic_model_metadata", "semantic_models", SEMANTIC_MODEL_SCHEMA),
    "best_practice_findings": ("semantic_model_best_practice", "semantic_model_best_practice_rule_findings", BEST_PRACTICE_FINDING_SCHEMA),
    "overview": ("semantic_model_optimization", "semantic_model_optimization_overview", OPTIMIZATION_OVERVIEW_SCHEMA),
    "opportunities": ("semantic_model_optimization", "semantic_model_optimization_opportunities", OPTIMIZATION_OPPORTUNITY_SCHEMA),
    "recommendations": ("semantic_model_optimization", "semantic_model_optimization_recommendations", OPTIMIZATION_RECOMMENDATION_SCHEMA),
    "findings": ("semantic_model_optimization", "semantic_model_optimization_findings", OPTIMIZATION_FINDING_SCHEMA),
    "opportunity_recommendation_links": ("semantic_model_optimization", "semantic_model_optimization_opportunity_recommendation_links", OPTIMIZATION_LINK_SCHEMA),
    "opportunity_finding_links": ("semantic_model_optimization", "semantic_model_optimization_opportunity_finding_links", OPTIMIZATION_LINK_SCHEMA),
    "column_storage": ("semantic_model_vertipaq", "semantic_model_column_storage", COLUMN_STORAGE_SCHEMA),
    "table_storage": ("semantic_model_vertipaq", "semantic_model_table_storage", TABLE_STORAGE_SCHEMA),
}

CURRENT_STATE_TABLES = tuple(
    logical_name for logical_name in CURATED_TABLES if logical_name != "analysis_runs"
)


def curated_table_name(logical_name):
    schema_name, physical_name, _ = CURATED_TABLES[logical_name]
    return f"{schema_name}.{physical_name}"


def ensure_curated_tables():
    for schema_name in BUSINESS_SCHEMAS:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`")
    for logical_name, (_, _, schema) in CURATED_TABLES.items():
        name = curated_table_name(logical_name)
        if not spark.catalog.tableExists(name):
            spark.createDataFrame([], schema).write.format("delta").mode("errorifexists").saveAsTable(name)
            continue
        existing_columns = {field.name.lower() for field in spark.table(name).schema.fields}
        missing_fields = [field for field in schema.fields if field.name.lower() not in existing_columns]
        if missing_fields:
            additions = ", ".join(
                f"`{field.name}` {field.dataType.simpleString().upper()}"
                for field in missing_fields
            )
            spark.sql(f"ALTER TABLE {name} ADD COLUMNS ({additions})")

    findings_name = curated_table_name("findings")
    # Keep the regex backslashes intact through Python and Spark SQL parsing.
    # A normal f-string turns ``\\s`` into ``\s`` before Spark sees it; Spark's
    # SQL string parser then drops that remaining slash and the regex becomes
    # ``s+``, which corrupts identifiers such as ``FactInternetSales``.
    spark.sql(fr"""
        UPDATE {findings_name}
        SET
            object_scope = CASE
                WHEN instr(lower(coalesce(affected_table_name, '')), 'datetabletemplate_') > 0
                  OR instr(lower(coalesce(affected_table_name, '')), 'localdatetable_') > 0
                  OR instr(lower(coalesce(affected_object_name, '')), 'datetabletemplate_') > 0
                  OR instr(lower(coalesce(affected_object_name, '')), 'localdatetable_') > 0
                    THEN 'Auto Date/Time (system)'
                WHEN upper(trim(coalesce(affected_object_type, ''))) IN ('', 'MODEL', 'SEMANTIC MODEL')
                    THEN 'Model-level'
                ELSE 'Authored / imported object'
            END,
            display_table_name = CASE
                WHEN trim(coalesce(affected_table_name, '')) <> '' THEN
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                trim(regexp_replace(
                                    replace(replace(replace(affected_table_name, chr(8203), ''), chr(65279), ''), chr(160), ' '),
                                    '\\s+', ' '
                                )),
                                '^\\x27+|\\x27+$', ''
                            ),
                            '^"+|"+$', ''
                        ),
                        '^`+|`+$', ''
                    )
                WHEN instr(coalesce(affected_object_name, ''), '[') > 0
                    THEN regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                trim(regexp_replace(
                                    replace(replace(replace(substring_index(affected_object_name, '[', 1), chr(8203), ''), chr(65279), ''), chr(160), ' '),
                                    '\\s+', ' '
                                )),
                                '^\\x27+|\\x27+$', ''
                            ),
                            '^"+|"+$', ''
                        ),
                        '^`+|`+$', ''
                    )
                WHEN upper(trim(coalesce(affected_object_type, ''))) IN ('TABLE', 'CALCULATED TABLE')
                  AND trim(coalesce(affected_object_name, '')) <> ''
                    THEN regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                trim(regexp_replace(
                                    replace(replace(replace(affected_object_name, chr(8203), ''), chr(65279), ''), chr(160), ' '),
                                    '\\s+', ' '
                                )),
                                '^\\x27+|\\x27+$', ''
                            ),
                            '^"+|"+$', ''
                        ),
                        '^`+|`+$', ''
                    )
                ELSE 'Not applicable'
            END,
            display_object_name = CASE
                WHEN trim(coalesce(affected_object_name, '')) = '' THEN 'Not applicable'
                ELSE trim(affected_object_name)
            END
    """)


def replace_semantic_model_current_state(logical_name, semantic_model_id, rows):
    _, _, schema = CURATED_TABLES[logical_name]
    name = curated_table_name(logical_name)
    escaped_model_id = semantic_model_id.replace("'", "''")
    DeltaTable.forName(spark, name).delete(f"semantic_model_id = '{escaped_model_id}'")
    if rows:
        spark.createDataFrame(rows, schema=schema).write.format("delta").mode("append").saveAsTable(name)


def reconcile_workspace_current_state(targets):
    """Remove current-state rows for models no longer eligible in a full workspace scan."""
    workspace_targets = {}
    for target in targets:
        if target.get("scope_source") != "WORKSPACE":
            continue
        workspace_targets.setdefault(target["workspace_id"], set()).add(target["model_id"])

    if not workspace_targets:
        return

    model_dimension = spark.table(curated_table_name("semantic_models"))
    stale_model_ids = set()
    for workspace_id, eligible_model_ids in workspace_targets.items():
        escaped_workspace_id = workspace_id.replace("'", "''")
        existing_model_ids = {
            row["semantic_model_id"]
            for row in (
                model_dimension
                .where(f"workspace_id = '{escaped_workspace_id}'")
                .select("semantic_model_id")
                .collect()
            )
        }
        stale_model_ids.update(existing_model_ids - eligible_model_ids)

    if not stale_model_ids:
        return

    quoted_ids = ", ".join(
        "'" + model_id.replace("'", "''") + "'"
        for model_id in sorted(stale_model_ids)
    )
    predicate = f"semantic_model_id IN ({quoted_ids})"
    for logical_name in CURRENT_STATE_TABLES:
        DeltaTable.forName(spark, curated_table_name(logical_name)).delete(predicate)
    print(
        f"Removed stale current-state rows for {len(stale_model_ids)} semantic model(s) "
        "outside the eligible full-workspace scan scope."
    )


def validate_curated_scan_output(model_results):
    """Reject false-success and internally inconsistent business-layer output."""
    analyzed_model_ids = sorted({
        row["model_id"]
        for row in model_results
        if row["overall_status"] in {"SUCCEEDED", "PARTIAL"}
    })
    if not analyzed_model_ids:
        return
    expected_result_by_model = {
        row["model_id"]: row
        for row in model_results
        if row["overall_status"] in {"SUCCEEDED", "PARTIAL"}
    }

    quoted_ids = ", ".join(
        "'" + model_id.replace("'", "''") + "'"
        for model_id in analyzed_model_ids
    )
    missing_by_table = {}
    for logical_name in ("analysis_runs", "semantic_models", "overview"):
        present_ids = {
            row["semantic_model_id"]
            for row in spark.sql(
                f"SELECT DISTINCT semantic_model_id "
                f"FROM {curated_table_name(logical_name)} "
                f"WHERE semantic_model_id IN ({quoted_ids})"
            ).collect()
        }
        missing_ids = sorted(set(analyzed_model_ids) - present_ids)
        if missing_ids:
            missing_by_table[curated_table_name(logical_name)] = missing_ids

    if missing_by_table:
        raise RuntimeError(
            "Scan did not materialize the required business-layer rows: "
            + json.dumps(missing_by_table, ensure_ascii=False)
        )

    def grouped_counts(logical_name, extra_expressions=()):
        expressions = ["COUNT(*) AS row_count", *extra_expressions]
        rows = spark.sql(
            "SELECT semantic_model_id, "
            + ", ".join(expressions)
            + f" FROM {curated_table_name(logical_name)}"
            + f" WHERE semantic_model_id IN ({quoted_ids})"
            + " GROUP BY semantic_model_id"
        ).collect()
        return {row["semantic_model_id"]: row.asDict(recursive=True) for row in rows}

    overview_by_model = {
        row["semantic_model_id"]: row.asDict(recursive=True)
        for row in spark.sql(
            "SELECT * FROM " + curated_table_name("overview")
            + f" WHERE semantic_model_id IN ({quoted_ids})"
        ).collect()
    }
    dimension_by_model = {
        row["semantic_model_id"]: row.asDict(recursive=True)
        for row in spark.sql(
            "SELECT semantic_model_id, latest_analysis_id, latest_analysis_status "
            "FROM " + curated_table_name("semantic_models")
            + f" WHERE semantic_model_id IN ({quoted_ids})"
        ).collect()
    }
    analysis_history_pairs = {
        (row["semantic_model_id"], row["analysis_id"])
        for row in spark.sql(
            "SELECT semantic_model_id, analysis_id FROM "
            + curated_table_name("analysis_runs")
            + f" WHERE semantic_model_id IN ({quoted_ids})"
        ).collect()
    }
    opportunity_counts = grouped_counts("opportunities")
    recommendation_counts = grouped_counts("recommendations", (
        "SUM(CASE WHEN actionability_status = 'ACTIONABLE' THEN 1 ELSE 0 END) AS actionable_count",
        "SUM(CASE WHEN actionability_status = 'REVIEW_REQUIRED' THEN 1 ELSE 0 END) AS review_required_count",
    ))
    finding_counts = grouped_counts("findings", (
        "SUM(CASE WHEN actionability_status = 'SUPPRESSED' THEN 1 ELSE 0 END) AS suppressed_count",
    ))
    recommendation_link_counts = grouped_counts("opportunity_recommendation_links")
    finding_link_counts = grouped_counts("opportunity_finding_links")

    consistency_issues = []
    for semantic_model_id in analyzed_model_ids:
        overview = overview_by_model[semantic_model_id]
        dimension = dimension_by_model[semantic_model_id]
        expected_result = expected_result_by_model[semantic_model_id]
        expected_counts = {
            "optimization_opportunity_count": opportunity_counts.get(semantic_model_id, {}).get("row_count", 0),
            "optimization_recommendation_count": recommendation_counts.get(semantic_model_id, {}).get("row_count", 0),
            "optimization_finding_count": finding_counts.get(semantic_model_id, {}).get("row_count", 0),
            "actionable_recommendation_count": recommendation_counts.get(semantic_model_id, {}).get("actionable_count", 0),
            "review_required_recommendation_count": recommendation_counts.get(semantic_model_id, {}).get("review_required_count", 0),
            "suppressed_finding_count": finding_counts.get(semantic_model_id, {}).get("suppressed_count", 0),
        }
        mismatches = {
            name: {"overview": overview.get(name), "detail": expected}
            for name, expected in expected_counts.items()
            if overview.get(name) != expected
        }
        if dimension.get("latest_analysis_id") != overview.get("analysis_id"):
            mismatches["latest_analysis_id"] = {
                "semantic_models": dimension.get("latest_analysis_id"),
                "overview": overview.get("analysis_id"),
            }
        if overview.get("analysis_id") != expected_result.get("scan_id"):
            mismatches["current_scan_analysis_id"] = {
                "expected": expected_result.get("scan_id"),
                "overview": overview.get("analysis_id"),
            }
        if (semantic_model_id, expected_result.get("scan_id")) not in analysis_history_pairs:
            mismatches["analysis_history"] = {
                "semantic_model_id": semantic_model_id,
                "analysis_id": expected_result.get("scan_id"),
                "status": "missing",
            }
        if dimension.get("latest_analysis_status") != overview.get("analysis_status"):
            mismatches["latest_analysis_status"] = {
                "semantic_models": dimension.get("latest_analysis_status"),
                "overview": overview.get("analysis_status"),
            }
        if overview.get("analysis_status") != expected_result.get("overall_status"):
            mismatches["current_scan_analysis_status"] = {
                "expected": expected_result.get("overall_status"),
                "overview": overview.get("analysis_status"),
            }
        if not str(overview.get("data_availability_explanation") or "").strip():
            mismatches["data_availability_explanation"] = "missing"
        recommendation_row_count = recommendation_counts.get(semantic_model_id, {}).get("row_count", 0)
        recommendation_link_count = recommendation_link_counts.get(semantic_model_id, {}).get("row_count", 0)
        if recommendation_link_count != recommendation_row_count:
            mismatches["opportunity_recommendation_link_count"] = {
                "links": recommendation_link_count,
                "recommendations": recommendation_row_count,
            }
        finding_row_count = finding_counts.get(semantic_model_id, {}).get("row_count", 0)
        finding_link_count = finding_link_counts.get(semantic_model_id, {}).get("row_count", 0)
        if finding_link_count != finding_row_count:
            mismatches["opportunity_finding_link_count"] = {
                "links": finding_link_count,
                "findings": finding_row_count,
            }
        if mismatches:
            consistency_issues.append({"semantic_model_id": semantic_model_id, "mismatches": mismatches})

    quality_checks = {
        "invalid_findings": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('findings')}
            WHERE semantic_model_id IN ({quoted_ids}) AND (
                actionability_status NOT IN ('ACTIONABLE', 'REVIEW_REQUIRED', 'INFORMATIONAL', 'SUPPRESSED')
                OR finding_priority_score IS NULL OR finding_priority_score < 0 OR finding_priority_score > 100
                OR finding_priority_band <> CASE
                    WHEN finding_priority_score >= 80 THEN 'P1_CRITICAL'
                    WHEN finding_priority_score >= 65 THEN 'P2_HIGH'
                    WHEN finding_priority_score >= 40 THEN 'P3_MEDIUM'
                    ELSE 'P4_LOW' END
                OR TRIM(COALESCE(actionability_reason, '')) = ''
                OR (actionability_status = 'SUPPRESSED' AND TRIM(COALESCE(suppression_reason, '')) = '')
            )
        """,
        "invalid_recommendations": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('recommendations')}
            WHERE semantic_model_id IN ({quoted_ids}) AND (
                actionability_status NOT IN ('ACTIONABLE', 'REVIEW_REQUIRED', 'INFORMATIONAL', 'SUPPRESSED')
                OR recommendation_priority_score IS NULL OR recommendation_priority_score < 0 OR recommendation_priority_score > 100
                OR recommendation_priority_band <> CASE
                    WHEN recommendation_priority_score >= 80 THEN 'P1_CRITICAL'
                    WHEN recommendation_priority_score >= 65 THEN 'P2_HIGH'
                    WHEN recommendation_priority_score >= 40 THEN 'P3_MEDIUM'
                    ELSE 'P4_LOW' END
                OR automation_eligibility NOT IN ('SCRIPT_CANDIDATE', 'MANUAL_ONLY', 'MANUAL_REVIEW', 'NOT_ELIGIBLE')
                OR TRIM(COALESCE(recommendation_title, '')) = ''
                OR TRIM(COALESCE(actionability_reason, '')) = ''
                OR TRIM(COALESCE(why_it_matters, '')) = ''
                OR TRIM(COALESCE(validation_method, '')) = ''
                OR TRIM(COALESCE(rollback_guidance, '')) = ''
                OR (actionability_status IN ('ACTIONABLE', 'REVIEW_REQUIRED') AND TRIM(COALESCE(recommended_action, '')) = '')
                OR (actionability_status IN ('INFORMATIONAL', 'SUPPRESSED') AND automation_eligibility <> 'NOT_ELIGIBLE')
                OR (actionability_status = 'REVIEW_REQUIRED' AND automation_eligibility <> 'MANUAL_REVIEW')
            )
        """,
        "invalid_opportunities": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('opportunities')}
            WHERE semantic_model_id IN ({quoted_ids}) AND (
                actionability_status NOT IN ('ACTIONABLE', 'REVIEW_REQUIRED', 'INFORMATIONAL', 'SUPPRESSED')
                OR priority_score IS NULL OR priority_score < 0 OR priority_score > 100
                OR priority_band <> CASE
                    WHEN priority_score >= 80 THEN 'P1_CRITICAL'
                    WHEN priority_score >= 65 THEN 'P2_HIGH'
                    WHEN priority_score >= 40 THEN 'P3_MEDIUM'
                    ELSE 'P4_LOW' END
                OR TRIM(COALESCE(opportunity_title, '')) = ''
                OR TRIM(COALESCE(opportunity_summary, '')) = ''
            )
        """,
        "invalid_opportunity_rollups": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('opportunities')} o
            LEFT JOIN (
                SELECT semantic_model_id, opportunity_id, COUNT(*) AS finding_count
                FROM {curated_table_name('findings')}
                GROUP BY semantic_model_id, opportunity_id
            ) f ON o.semantic_model_id = f.semantic_model_id AND o.opportunity_id = f.opportunity_id
            LEFT JOIN (
                SELECT semantic_model_id, opportunity_id, COUNT(*) AS recommendation_count
                FROM {curated_table_name('recommendations')}
                GROUP BY semantic_model_id, opportunity_id
            ) r ON o.semantic_model_id = r.semantic_model_id AND o.opportunity_id = r.opportunity_id
            WHERE o.semantic_model_id IN ({quoted_ids}) AND (
                o.finding_count <> COALESCE(f.finding_count, 0)
                OR o.recommendation_count <> COALESCE(r.recommendation_count, 0)
            )
        """,
        "orphan_recommendations": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('recommendations')} r
            LEFT ANTI JOIN {curated_table_name('opportunities')} o
              ON r.semantic_model_id = o.semantic_model_id AND r.opportunity_id = o.opportunity_id
            WHERE r.semantic_model_id IN ({quoted_ids})
        """,
        "orphan_findings": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('findings')} f
            LEFT ANTI JOIN {curated_table_name('opportunities')} o
              ON f.semantic_model_id = o.semantic_model_id AND f.opportunity_id = o.opportunity_id
            WHERE f.semantic_model_id IN ({quoted_ids})
        """,
        "invalid_recommendation_links": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('opportunity_recommendation_links')} l
            LEFT ANTI JOIN {curated_table_name('recommendations')} r
              ON l.semantic_model_id = r.semantic_model_id
             AND l.opportunity_id = r.opportunity_id
             AND l.related_entity_id = r.recommendation_id
            WHERE l.semantic_model_id IN ({quoted_ids})
        """,
        "invalid_finding_links": f"""
            SELECT COUNT(*) AS issue_count
            FROM {curated_table_name('opportunity_finding_links')} l
            LEFT ANTI JOIN {curated_table_name('findings')} f
              ON l.semantic_model_id = f.semantic_model_id
             AND l.opportunity_id = f.opportunity_id
             AND l.related_entity_id = f.finding_id
            WHERE l.semantic_model_id IN ({quoted_ids})
        """,
    }
    failed_quality_checks = {}
    for name, query in quality_checks.items():
        issue_count = spark.sql(query).first()["issue_count"]
        if issue_count:
            failed_quality_checks[name] = issue_count
    if consistency_issues or failed_quality_checks:
        raise RuntimeError(
            "Business-layer quality validation failed: "
            + json.dumps({
                "consistency_issues": consistency_issues,
                "failed_quality_checks": failed_quality_checks,
            }, ensure_ascii=False)
        )


def upsert_curated_history(logical_name, rows, keys):
    if not rows:
        return
    _, _, schema = CURATED_TABLES[logical_name]
    source = spark.createDataFrame(rows, schema=schema)
    condition = " AND ".join(f"t.`{key}` = s.`{key}`" for key in keys)
    (
        DeltaTable.forName(spark, curated_table_name(logical_name))
        .alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def severity_value(value):
    return {
        "CRITICAL": 5,
        "ERROR": 4,
        "HIGH": 4,
        "WARNING": 3,
        "MEDIUM": 3,
        "INFO": 2,
        "LOW": 1,
    }.get((value or "").upper(), 0)


def normalize_display_identifier(value):
    normalized = str(value or "").replace("\u200b", "").replace("\ufeff", "").replace("\u00a0", " ")
    normalized = re.sub(r"\s+", " ", normalized).strip()
    while len(normalized) >= 2 and normalized[0] == normalized[-1] and normalized[0] in "'\"`":
        normalized = normalized[1:-1].strip()
    return normalized


def finding_locator_fields(finding):
    raw_table = (finding.get("table_name") or "").strip()
    raw_object = (finding.get("object_name") or "").strip()
    object_type = (finding.get("object_type") or "").strip().upper()
    auto_date = any(
        marker in value.lower()
        for marker in ("datetabletemplate_", "localdatetable_")
        for value in (raw_table, raw_object)
    )
    object_scope = (
        "Auto Date/Time (system)" if auto_date
        else "Model-level" if object_type in {"", "MODEL", "SEMANTIC MODEL"}
        else "Authored / imported object"
    )
    object_table = ""
    if "[" in raw_object:
        object_table = normalize_display_identifier(raw_object.split("[", 1)[0])
    display_table = normalize_display_identifier(raw_table) or object_table
    if not display_table and object_type in {"TABLE", "CALCULATED TABLE"}:
        display_table = normalize_display_identifier(raw_object)
    display_table = display_table or "Not applicable"
    return {
        "object_scope": object_scope,
        "display_table_name": display_table,
        "display_object_name": raw_object or "Not applicable",
    }


def risk_value(value):
    return {"HIGH": 3, "MEDIUM": 2, "LOW": 1}.get((value or "").upper(), 0)


def availability_explanations(model_row, result):
    notes = []
    bpa_findings = [
        row for row in result["findings"]
        if (row.get("source") or "").upper() == "BPA"
    ]
    if model_row["bpa_status"] == "SUCCEEDED" and not bpa_findings:
        notes.append("Best-practice analysis: completed with no rule violations.")
    elif model_row["bpa_status"] == "NOT_RUN":
        notes.append("Best-practice analysis: not requested by this analysis profile.")
    elif model_row["bpa_status"] == "FAILED":
        notes.append("Best-practice analysis: failed; see analysis run error details.")
    if model_row["vpa_status"] == "SUCCEEDED" and not result["vpa_columns"] and not result["vpa_tables"]:
        notes.append("Storage analysis: completed with no column or table storage records.")
    elif model_row["vpa_status"] == "NOT_RUN":
        notes.append("Storage analysis: not requested by this analysis profile.")
    elif model_row["vpa_status"] == "FAILED":
        notes.append("Storage analysis: failed; see analysis run error details.")
    if model_row["refresh_status"] == "SUCCEEDED" and not result["refresh_rows"]:
        notes.append("Refresh history: no records were returned for the selected history window.")
    elif model_row["refresh_status"] == "NOT_RUN":
        notes.append("Refresh history: not requested by this analysis profile.")
    elif model_row["refresh_status"] == "FAILED":
        notes.append("Refresh history: failed; see analysis run error details.")
    if model_row["usage_status"] == "NOT_RUN":
        notes.append("Object usage: not run in the standard profile; use the deep profile to collect usage evidence.")
    elif model_row["usage_status"] == "SUCCEEDED" and not result["usage_rows"]:
        notes.append("Object usage: analysis completed and returned no observations.")
    elif model_row["usage_status"] == "FAILED":
        notes.append("Object usage: failed; see analysis run error details.")
    if model_row["direct_lake_status"] == "NOT_APPLICABLE":
        notes.append(
            "Direct Lake checks: not applicable to storage mode "
            + str(model_row.get("storage_mode") or "UNKNOWN")
            + "."
        )
    elif model_row["direct_lake_status"] == "SUCCEEDED" and not result["direct_lake_rows"]:
        notes.append("Direct Lake checks: completed with no fallback observations.")
    elif model_row["direct_lake_status"] == "NOT_RUN":
        notes.append("Direct Lake checks: not requested by this analysis profile.")
    elif model_row["direct_lake_status"] == "FAILED":
        notes.append("Direct Lake checks: failed; see analysis run error details.")
    if model_row["access_snapshot_status"] == "SUCCEEDED" and not result["access_rows"]:
        notes.append("Item access snapshot: completed and returned no explicit access records.")
    elif model_row["access_snapshot_status"] == "NOT_APPLICABLE_WORKSPACE_USER_PROFILE":
        notes.append("Item access snapshot: not applicable to the normal workspace-user profile.")
    elif model_row["access_snapshot_status"] == "NOT_RUN":
        notes.append("Item access snapshot: not requested by this analysis profile.")
    elif model_row["access_snapshot_status"] == "FAILED":
        notes.append("Item access snapshot: failed optional governance enrichment; see analysis run error details.")
    return " ".join(notes) or "All requested evidence sources returned data or an explicit status."


def curate_latest_model_analysis(result):
    model_row = result["model_row"]
    analysis_run_rows = [{
        "analysis_id": model_row["scan_id"],
        "workspace_id": model_row["workspace_id"],
        "workspace_name": model_row["workspace_name"],
        "semantic_model_id": model_row["model_id"],
        "semantic_model_name": model_row["model_name"],
        "scanner_version": model_row["scanner_version"],
        "analysis_profile": analysis_profile,
        "analysis_status": model_row["overall_status"],
        "permission_precheck_status": model_row["permission_precheck_status"],
        "best_practice_analysis_status": model_row["bpa_status"],
        "storage_analysis_status": model_row["vpa_status"],
        "refresh_history_status": model_row["refresh_status"],
        "object_usage_analysis_status": model_row["usage_status"],
        "direct_lake_analysis_status": model_row["direct_lake_status"],
        "item_access_snapshot_status": model_row["access_snapshot_status"],
        "finding_count": model_row["finding_count"],
        "started_at": model_row["started_at"],
        "completed_at": model_row["completed_at"],
        "duration_seconds": model_row["duration_seconds"],
        "error_details": model_row["error_json"],
    }]
    upsert_curated_history("analysis_runs", analysis_run_rows, ["analysis_id", "semantic_model_id"])
    if model_row["overall_status"] not in {"SUCCEEDED", "PARTIAL"}:
        return

    analysis_id = model_row["scan_id"]
    semantic_model_id = model_row["model_id"]
    workspace_name = model_row["workspace_name"]
    semantic_model_name = model_row["model_name"]
    findings = result["findings"]
    auto_date_present = any(is_auto_date_root_cause_finding(row) for row in findings)

    opportunity_groups = {}
    recommendation_groups = {}
    finding_rows = []
    finding_links = []
    recommendation_links = set()

    for finding in findings:
        finding_quality = grade_finding(finding)
        raw_domain = finding.get("category") or finding.get("impact_area") or "General optimization"
        raw_source = finding.get("source") or "Unknown source"
        consolidation = root_cause_grouping(finding, auto_date_present)
        domain = consolidation["domain"] if consolidation else raw_domain
        source = consolidation["source"] if consolidation else raw_source
        grouping_key = consolidation["key"] if consolidation else f"{source}|{domain}"
        opportunity_id = stable_id(semantic_model_id, "OPPORTUNITY", grouping_key)
        opportunity = opportunity_groups.setdefault(opportunity_id, {
            "findings": [],
            "recommendations": set(),
            "domain": domain,
            "source": source,
            "title": consolidation["title"] if consolidation else f"{domain} optimization",
        })
        opportunity["findings"].append(finding)

        recommendation_id = stable_id(
            semantic_model_id,
            "RECOMMENDATION",
            opportunity_id,
            consolidation["key"] if consolidation else finding.get("rule_id") or finding.get("rule_name"),
            consolidation["action"] if consolidation else finding.get("recommended_action"),
        )
        recommendation = recommendation_groups.setdefault(recommendation_id, {
            "findings": [],
            "opportunity_id": opportunity_id,
            "domain": domain,
            "source": source,
            "title": consolidation["title"] if consolidation else finding.get("rule_name") or "Optimization recommendation",
            "action": consolidation["action"] if consolidation else finding.get("recommended_action"),
        })
        recommendation["findings"].append(finding)
        opportunity["recommendations"].add(recommendation_id)
        recommendation_links.add((opportunity_id, recommendation_id))
        finding_links.append({
            "analysis_id": analysis_id,
            "semantic_model_id": semantic_model_id,
            "opportunity_id": opportunity_id,
            "related_entity_id": finding["finding_id"],
        })
        finding_rows.append({
            "finding_id": finding["finding_id"],
            "opportunity_id": opportunity_id,
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "finding_source": raw_source,
            "optimization_domain": raw_domain,
            "rule_name": finding.get("rule_name"),
            "severity": finding.get("severity"),
            "confidence": finding.get("confidence"),
            "impact_area": finding.get("impact_area"),
            "affected_object_type": finding.get("object_type"),
            "affected_table_name": finding.get("table_name"),
            "affected_object_name": finding.get("object_name"),
            **finding_locator_fields(finding),
            "finding_description": finding.get("finding_text"),
            "recommended_action": finding.get("recommended_action"),
            "technical_evidence": finding.get("technical_evidence"),
            "estimated_saving_bytes_low": finding.get("estimated_saving_bytes_low"),
            "estimated_saving_bytes_high": finding.get("estimated_saving_bytes_high"),
            "change_risk": finding.get("change_risk"),
            "validation_required": finding.get("validation_required"),
            **finding_quality,
            "detected_at": finding.get("detected_at"),
        })

    opportunity_rows = []
    for opportunity_id, group in opportunity_groups.items():
        grouped_findings = group["findings"]
        highest = max(grouped_findings, key=lambda row: severity_value(row.get("severity")))
        highest_risk = max(grouped_findings, key=lambda row: risk_value(row.get("change_risk")))
        opportunity_quality = grade_opportunity(grouped_findings)
        opportunity_rows.append({
            "opportunity_id": opportunity_id,
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "opportunity_title": group["title"],
            "optimization_domain": group["domain"],
            "finding_source": group["source"],
            "highest_severity": highest.get("severity"),
            "finding_count": len(grouped_findings),
            "recommendation_count": len(group["recommendations"]),
            "estimated_saving_bytes_low": sum(row.get("estimated_saving_bytes_low") or 0 for row in grouped_findings),
            "estimated_saving_bytes_high": sum(row.get("estimated_saving_bytes_high") or 0 for row in grouped_findings),
            "change_risk": highest_risk.get("change_risk"),
            "opportunity_summary": summarize_opportunity(
                grouped_findings, group["source"], group["domain"]
            ),
            **opportunity_quality,
            "detected_at": max(row.get("detected_at") for row in grouped_findings if row.get("detected_at")),
        })

    recommendation_rows = []
    for recommendation_id, group in recommendation_groups.items():
        grouped_findings = group["findings"]
        highest_risk = max(grouped_findings, key=lambda row: risk_value(row.get("change_risk")))
        recommendation_quality = grade_recommendation(
            grouped_findings, group["domain"], group["title"], group["action"]
        )
        recommendation_title = recommendation_quality.pop("recommendation_title")
        recommended_action = recommendation_quality.pop("recommended_action")
        recommendation_rows.append({
            "recommendation_id": recommendation_id,
            "opportunity_id": group["opportunity_id"],
            "analysis_id": analysis_id,
            "workspace_name": workspace_name,
            "semantic_model_id": semantic_model_id,
            "semantic_model_name": semantic_model_name,
            "recommendation_title": recommendation_title,
            "optimization_domain": group["domain"],
            "recommended_action": recommended_action,
            "change_risk": highest_risk.get("change_risk"),
            "validation_required": any(row.get("validation_required") for row in grouped_findings),
            "estimated_saving_bytes_low": sum(row.get("estimated_saving_bytes_low") or 0 for row in grouped_findings),
            "estimated_saving_bytes_high": sum(row.get("estimated_saving_bytes_high") or 0 for row in grouped_findings),
            "finding_source": group["source"],
            "affected_finding_count": len(grouped_findings),
            **recommendation_quality,
            "detected_at": max(row.get("detected_at") for row in grouped_findings if row.get("detected_at")),
        })

    recommendation_link_rows = [
        {
            "analysis_id": analysis_id,
            "semantic_model_id": semantic_model_id,
            "opportunity_id": opportunity_id,
            "related_entity_id": recommendation_id,
        }
        for opportunity_id, recommendation_id in sorted(recommendation_links)
    ]

    column_storage_rows = [{
        "column_storage_record_id": row["evidence_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "table_name": row.get("table_name"),
        "column_name": row.get("column_name"),
        "data_type": row.get("data_type"),
        "encoding": row.get("encoding"),
        "cardinality": row.get("cardinality"),
        "data_size_bytes": row.get("data_size_bytes"),
        "dictionary_size_bytes": row.get("dictionary_size_bytes"),
        "hierarchy_size_bytes": row.get("hierarchy_size_bytes"),
        "total_size_bytes": row.get("total_size_bytes"),
        "percentage_of_semantic_model_size": row.get("model_size_pct"),
        "detected_at": row.get("detected_at"),
    } for row in result["vpa_columns"]]

    table_storage_rows = [{
        "table_storage_record_id": row["evidence_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "table_name": row.get("table_name"),
        "row_count": row.get("row_count"),
        "data_size_bytes": row.get("data_size_bytes"),
        "dictionary_size_bytes": row.get("dictionary_size_bytes"),
        "hierarchy_size_bytes": row.get("hierarchy_size_bytes"),
        "total_size_bytes": row.get("total_size_bytes"),
        "percentage_of_semantic_model_size": row.get("model_size_pct"),
        "detected_at": row.get("detected_at"),
    } for row in result["vpa_tables"]]

    best_practice_rows = [{
        "best_practice_finding_id": finding["finding_id"],
        "analysis_id": analysis_id,
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "rule_id": finding.get("rule_id"),
        "rule_name": finding.get("rule_name"),
        "category": finding.get("category"),
        "severity": finding.get("severity"),
        "affected_object_type": finding.get("object_type"),
        "affected_table_name": finding.get("table_name"),
        "affected_object_name": finding.get("object_name"),
        "finding_description": finding.get("finding_text"),
        "recommended_action": finding.get("recommended_action"),
        "technical_evidence": finding.get("technical_evidence"),
        "documentation_url": finding.get("documentation_url"),
        "detected_at": finding.get("detected_at"),
    } for finding in findings if (finding.get("source") or "").upper() == "BPA"]

    semantic_model_rows = [{
        "semantic_model_id": semantic_model_id,
        "workspace_id": model_row["workspace_id"],
        "workspace_name": workspace_name,
        "semantic_model_name": semantic_model_name,
        "capacity_id": model_row["capacity_id"],
        "capacity_name": model_row["capacity_name"],
        "storage_mode": model_row["storage_mode"],
        "semantic_model_size_bytes": model_row["model_size_bytes"],
        "latest_analysis_id": analysis_id,
        "latest_analysis_status": model_row["overall_status"],
        "latest_analysis_at": model_row["completed_at"],
        "scanner_version": model_row["scanner_version"],
    }]

    overview_rows = [{
        "analysis_id": analysis_id,
        "workspace_id": model_row["workspace_id"],
        "workspace_name": workspace_name,
        "semantic_model_id": semantic_model_id,
        "semantic_model_name": semantic_model_name,
        "storage_mode": model_row["storage_mode"],
        "analysis_status": model_row["overall_status"],
        "analysis_completed_at": model_row["completed_at"],
        "scanner_version": model_row["scanner_version"],
        "semantic_model_size_bytes": model_row["model_size_bytes"],
        "optimization_opportunity_count": len(opportunity_rows),
        "optimization_recommendation_count": len(recommendation_rows),
        "optimization_finding_count": len(finding_rows),
        "high_severity_finding_count": sum((row.get("severity") or "").upper() in {"HIGH", "CRITICAL", "ERROR"} for row in findings),
        "actionable_recommendation_count": sum(row["actionability_status"] == ACTIONABLE for row in recommendation_rows),
        "review_required_recommendation_count": sum(row["actionability_status"] == REVIEW_REQUIRED for row in recommendation_rows),
        "suppressed_finding_count": sum(row["actionability_status"] == SUPPRESSED for row in finding_rows),
        "best_practice_analysis_status": model_row["bpa_status"],
        "storage_analysis_status": model_row["vpa_status"],
        "refresh_history_status": model_row["refresh_status"],
        "refresh_history_record_count": len(result["refresh_rows"]),
        "object_usage_analysis_status": model_row["usage_status"],
        "object_usage_observation_count": len(result["usage_rows"]),
        "direct_lake_analysis_status": model_row["direct_lake_status"],
        "direct_lake_observation_count": len(result["direct_lake_rows"]),
        "item_access_snapshot_status": model_row["access_snapshot_status"],
        "item_access_record_count": len(result["access_rows"]),
        "data_availability_explanation": availability_explanations(model_row, result),
    }]

    replacements = {
        "semantic_models": semantic_model_rows,
        "best_practice_findings": best_practice_rows,
        "overview": overview_rows,
        "opportunities": opportunity_rows,
        "recommendations": recommendation_rows,
        "findings": finding_rows,
        "opportunity_recommendation_links": recommendation_link_rows,
        "opportunity_finding_links": finding_links,
        "column_storage": column_storage_rows,
        "table_storage": table_storage_rows,
    }
    for logical_name, rows in replacements.items():
        replace_semantic_model_current_state(logical_name, semantic_model_id, rows)


In [ ]:
# ---------- Execute scan and persist each model immediately ----------

run_error = None
model_results = []
targets = []

try:
    ensure_tables()
    ensure_curated_tables()
    if initialize_only:
        init_summary = {"status": "INITIALIZED", "raw_table_count": len(TABLES), "curated_table_count": len(CURATED_TABLES)}
        print(json.dumps(init_summary, indent=2))
    else:
        with authentication_context():
            targets = resolve_targets()
            if not targets:
                raise RuntimeError(
                    "No eligible semantic models were resolved for the requested scan scope."
                )
            print(
                f"Resolved {len(targets)} semantic model(s) across "
                f"{len({target['workspace_id'] for target in targets})} workspace(s)."
            )
            display(pd.DataFrame([
                {
                    "Workspace": target["workspace_name"],
                    "Semantic model": target["model_name"],
                    "Permission check": target["permission_precheck_status"],
                }
                for target in targets
            ]))
    
            run_row = {
                "scan_id": SCAN_ID,
                "request_id": REQUEST_ID,
                "requested_by_upn": requested_by_upn.strip().lower() or None,
                "auth_mode": auth_mode,
                "analysis_profile": analysis_profile,
                "scanner_version": SCANNER_VERSION,
                "solution_stage": SOLUTION_STAGE,
                "status": "RUNNING",
                "started_at": RUN_STARTED_AT,
                "completed_at": None,
                "target_count": len(targets),
                "success_count": 0,
                "partial_count": 0,
                "failed_count": 0,
                "skipped_count": 0,
                "semantic_link_version": SEMANTIC_LINK_VERSION,
                "semantic_link_labs_version": SEMANTIC_LINK_LABS_VERSION,
                "parameters_hash": safe_parameters_hash(),
                "error_category": None,
                "error_message": None,
            }
            upsert_rows("scan_run", [run_row])
    
            sync_explicit_model_access(targets)
    
            for target in targets:
                result = scan_one_model(target)
                model_results.append(result["model_row"])
                upsert_rows("model_scan", [result["model_row"]])
                upsert_rows("dim_model", [result["dim_model_row"]])
                upsert_rows("finding", result["findings"])
                upsert_rows("vpa_column", result["vpa_columns"])
                upsert_rows("vpa_table", result["vpa_tables"])
                upsert_rows("object_usage", result["usage_rows"])
                upsert_rows("refresh", result["refresh_rows"])
                upsert_rows("direct_lake", result["direct_lake_rows"])
                upsert_rows("item_access_snapshot", result["access_rows"])
                curate_latest_model_analysis(result)
            reconcile_workspace_current_state(targets)
            validate_curated_scan_output(model_results)

except Exception as exc:
    run_error = exc
    print(traceback.format_exc())

completed_at = utcnow()
success_count = sum(row["overall_status"] == "SUCCEEDED" for row in model_results)
partial_count = sum(row["overall_status"] == "PARTIAL" for row in model_results)
failed_count = sum(row["overall_status"] == "FAILED" for row in model_results)
skipped_count = sum(row["overall_status"] == "SKIPPED_PERMISSION" for row in model_results)
model_failure_details = [
    f"{row['workspace_name']} / {row['model_name']}: {row.get('error_json') or 'core analyses failed without a captured component error'}"
    for row in model_results
    if row["overall_status"] == "FAILED"
]
model_failure_error = (
    clean_string("Model analysis failures: " + " | ".join(model_failure_details), 4000)
    if model_failure_details
    else None
)

if initialize_only:
    final_status = "INITIALIZED"
elif run_error is not None:
    final_status = "FAILED"
elif failed_count and fail_pipeline_if_any_model_fails:
    final_status = "FAILED"
elif skipped_count and fail_pipeline_if_permission_precheck_fails:
    final_status = "FAILED"
elif model_results and skipped_count == len(model_results):
    final_status = "SKIPPED_PERMISSION"
elif failed_count or partial_count or skipped_count:
    final_status = "PARTIAL"
else:
    final_status = "SUCCEEDED"

final_run_row = {
    "scan_id": SCAN_ID,
    "request_id": REQUEST_ID,
    "requested_by_upn": requested_by_upn.strip().lower() or None,
    "auth_mode": auth_mode,
    "analysis_profile": analysis_profile,
    "scanner_version": SCANNER_VERSION,
    "solution_stage": SOLUTION_STAGE,
    "status": final_status,
    "started_at": RUN_STARTED_AT,
    "completed_at": completed_at,
    "target_count": len(targets),
    "success_count": success_count,
    "partial_count": partial_count,
    "failed_count": failed_count,
    "skipped_count": skipped_count,
    "semantic_link_version": SEMANTIC_LINK_VERSION,
    "semantic_link_labs_version": SEMANTIC_LINK_LABS_VERSION,
    "parameters_hash": safe_parameters_hash(),
    "error_category": (
        error_category(run_error)
        if run_error
        else "MODEL_ANALYSIS"
        if model_failure_error
        else "AUTHORIZATION"
        if skipped_count and fail_pipeline_if_permission_precheck_fails
        else None
    ),
    "error_message": (
        truncate_error(run_error)
        if run_error
        else model_failure_error
        if model_failure_error
        else f"{skipped_count} model(s) skipped because the SPN workspace access precheck did not pass."
        if skipped_count and fail_pipeline_if_permission_precheck_fails
        else None
    ),
}

# If initialization failed before table creation, this write may also fail; keep the original error visible.
try:
    if not initialize_only and spark.catalog.tableExists(table_name("scan_run")):
        upsert_rows("scan_run", [final_run_row])
except Exception as persistence_exc:
    print(f"Unable to persist final run status: {truncate_error(persistence_exc)}")

summary = {
    "scan_id": SCAN_ID,
    "request_id": REQUEST_ID,
    "status": final_status,
    "target_count": len(targets),
    "success_count": success_count,
    "partial_count": partial_count,
    "failed_count": failed_count,
    "skipped_count": skipped_count,
    "duration_seconds": round((completed_at - RUN_STARTED_AT).total_seconds(), 2),
    "model_results": [
        {
            "workspace_id": row["workspace_id"],
            "model_id": row["model_id"],
            "model_name": row["model_name"],
            "status": row["overall_status"],
            "permission_precheck_status": row["permission_precheck_status"],
            "scanner_workspace_role": row["scanner_workspace_role"],
            "best_practice_analysis_status": row["bpa_status"],
            "storage_analysis_status": row["vpa_status"],
            "model_metadata_analysis_status": row.get("metadata_status"),
            "refresh_history_status": row["refresh_status"],
            "direct_lake_analysis_status": row["direct_lake_status"],
            "finding_count": row["finding_count"],
            "component_errors": row["error_json"],
        }
        for row in model_results
    ],
    "error": truncate_error(run_error) if run_error else model_failure_error,
}
print(
    f"Scan {final_status}: {success_count} succeeded, {partial_count} partial, "
    f"{failed_count} failed, {skipped_count} skipped. Scan ID: {SCAN_ID}"
)
if summary["model_results"]:
    display(pd.DataFrame(summary["model_results"]))
print(json.dumps(summary, indent=2))

if (
    run_error is not None
    or (failed_count and fail_pipeline_if_any_model_fails)
    or (skipped_count and fail_pipeline_if_permission_precheck_fails)
):
    raise RuntimeError(json.dumps(summary)) from run_error

if exit_notebook_with_summary:
    notebookutils.notebook.exit(json.dumps(summary))
